# Vector subtraction with a grid-stride loop

In the previous problem the grid grew with the input: one thread per element. Production kernels usually launch a **fixed** grid sized to the GPU (a few blocks per streaming multiprocessor) and let each thread loop over the array with a stride equal to the total number of threads. This is the **grid-stride loop**.

Write `vector_sub(a, b)` returning $\text{out}_i = a_i - b_i$.

## Inputs
- `a`, `b`: 1-D `float32` arrays of the same length `n` (`n` may be 0 or much larger than the grid).

## Launch configuration (fixed)
- Exactly **4 blocks of 64 threads** (256 threads total), whatever `n` is.
- Thread `t = cuda.grid(1)` handles elements `t, t + S, t + 2S, ...` while they are `< n`, where `S = cuda.gridsize(1)` is the total thread count.

## Output
A `float32` array of shape `(n,)`. Because the grid no longer depends on `n`, the `n == 0` case needs no special handling: every thread's loop simply runs zero times.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import numpy as np
from numba import cuda

BLOCKS, TPB = 4, 64

def vector_sub(a, b):
    """a, b: float32 (n,). Returns a - b as float32 (n,), using a fixed 4 x 64 grid."""
    d_a, d_b = cuda.to_device(a), cuda.to_device(b)
    d_out = cuda.device_array(a.shape[0], dtype=np.float32)
    sub_kernel[BLOCKS, TPB](d_a, d_b, d_out)
    return d_out.copy_to_host()

@cuda.jit
def sub_kernel(a, b, out):
    i = cuda.grid(1)
    # TODO: this only covers the first 256 elements. Loop with a stride of cuda.gridsize(1).
    if i < out.shape[0]:
        out[i] = a[i] - b[i]


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [5.0, 3.0, 1.0], "dtype": "float32"}, {"__nd__": [1.0, 1.0, 1.0], "dtype": "float32"}], "e": {"__nd__": [4.0, 2.0, 0.0], "dtype": "float32"}}, {"a": [{"__nd__": [0.5, -2.0], "dtype": "float32"}, {"__nd__": [-0.5, -2.0], "dtype": "float32"}], "e": {"__nd__": [1.0, 0.0], "dtype": "float32"}}, {"a": [{"__nd__": [], "dtype": "float32"}, {"__nd__": [], "dtype": "float32"}], "e": {"__nd__": [], "dtype": "float32"}}, {"a": [{"__nd__": [0.18905338644981384, -0.5227484703063965, -0.41306355595588684, -2.44146728515625, 1.7997074127197266, 1.1441658735275269, -0.32542282342910767, 0.7738065719604492, 0.28121066093444824, -0.5538228154182434, 0.9775674343109131, -0.310556560754776, -0.32882389426231384, -0.7921467423439026, 0.45495808124542236, -0.09919805079698563, 0.5452887415885925, -0.6071857213973999, 0.12682785093784332, -0.892274022102356, 0.8414649963378906, 0.18803508579730988, 0.3305709958076477, 0.41050392389297485, -1.0107574462890625, 0.7831810116767883, 2.0567028522491455, -1.6384425163269043, -1.7294114828109741, -1.5048314332962036, 0.8414589166641235, 0.1287156641483307, 1.0783424377441406, 0.7224308848381042, 0.21057181060314178, 0.28403815627098083, -0.16976049542427063, 0.8684602379798889, -1.129715919494629, -0.4218588173389435, 0.2429388463497162, 1.80142080783844, -0.7644641399383545, -1.079060435295105, -0.5632871985435486, 0.9692721962928772, -0.23500551283359528, 1.3243470191955566, -1.8725295066833496, 1.1285231113433838, 1.034866213798523, -1.4186656475067139, 0.1535818874835968, 1.2157576084136963, 0.08791904151439667, 0.9997014403343201, 2.374886989593506, 0.27393224835395813, -0.28038230538368225, -0.7710521817207336, 0.6480646133422852, -0.19673006236553192, -0.17874637246131897, -0.10525977611541748, 0.6498696804046631, -1.0663396120071411, -1.529876470565796, -2.4338767528533936, 1.1986708641052246, 0.07379335910081863, 1.5101194381713867, -0.008956298232078552, -0.7421554327011108, 0.4779292941093445, -0.07658841460943222, -1.2541866302490234, -0.885066568851471, 1.7667793035507202, 0.35435059666633606, 0.41638654470443726, -0.2765517234802246, -0.689719557762146, 0.8916558027267456, -0.10462645441293716, -0.7591081261634827, -0.13412852585315704, -0.9056060314178467, 0.19004583358764648, 1.129227638244629, -0.8360851407051086, 1.428544044494629, -0.6676199436187744, 0.15341216325759888, -0.836386501789093, -0.22223517298698425, 0.04740441218018532, -0.4346849322319031, -0.7028878927230835, -0.6778861880302429, -0.8211599588394165, -1.570080041885376, -0.26296961307525635, 0.4011582136154175, 0.9084019064903259, 0.6471027135848999, 2.45733642578125, 0.31868162751197815, -0.4563339054584503, 1.871634602546692, -1.0472180843353271, 0.9684776067733765, -0.9551447629928589, 0.3541116714477539, -1.968396782875061, 0.899273693561554, -0.15824761986732483, -0.9676814675331116, 1.6784189939498901, 0.7653546929359436, 0.04580783471465111, -0.7454434037208557, -0.04336259141564369, -0.16392876207828522, 0.7247765064239502, 0.7980751991271973, -0.6677563786506653, -0.5493705868721008, -0.5320653319358826, -1.349317193031311, -0.5915749073028564, -0.09240806847810745, 0.690951943397522, 1.3203794956207275, -0.8083458542823792, 0.5509765148162842, -0.4444568157196045, 2.0751876831054688, -0.04872354492545128, 0.5021269917488098, -0.9361621737480164, -0.8106894493103027, 0.2016953080892563, -0.3840544521808624, 0.35328859090805054, -1.561824083328247, 0.6620049476623535, -0.8994803428649902, 1.7224234342575073, -0.2884672284126282, 1.0972650051116943, -1.3949832916259766, 0.42894625663757324, -0.8880296349525452, -0.4868890643119812, 0.024596381932497025, -0.3170749545097351, 0.2642556130886078, 0.74765545129776, 0.652782142162323, -0.08748148381710052, -1.3634305000305176, -0.8046461343765259, -0.1957562118768692, -2.121825933456421, 0.7520805597305298, -0.23963819444179535, -0.2513192892074585, 0.9483762383460999, 0.6679105162620544, 0.22100096940994263, -1.0013335943222046, 0.31042158603668213, 0.3461775481700897, -0.884773313999176, 1.0433118343353271, 0.5078871846199036, -1.969624400138855, 0.1219494491815567, -1.2646980285644531, 1.136315107345581, 0.8070279359817505, -1.103590726852417, -0.8848726749420166, 0.10185328125953674, -0.23881082236766815, 1.5252031087875366, 0.7409738302230835, -0.28242725133895874, 0.5773959159851074, -2.0322792530059814, 0.30986469984054565, 0.866407036781311, -0.17895251512527466, -0.7102141976356506, 0.5658334493637085, 1.958614706993103, 0.20486892759799957, -1.1206777095794678, 1.4458048343658447, -1.8785474300384521, 0.503204882144928, -0.5819546580314636, 0.9305984377861023, -0.8494743704795837, 1.323847770690918, 0.3002730906009674, -1.726840615272522, -1.9738130569458008, -0.3160955011844635, 1.2942116260528564, 0.12703002989292145, 0.06712070107460022, -0.23590382933616638, 0.7379001975059509, 0.3773267865180969, 0.2803458869457245, -0.984722912311554, 1.1878533363342285, 1.363080382347107, -1.3013535737991333, 2.30149507522583, 0.722506582736969, -0.48800599575042725, -1.9043558835983276, 0.6568464040756226, 0.5575113892555237, -0.5956318378448486, -1.7905532121658325, 1.1244767904281616, -0.588318943977356, -0.4717450737953186, 2.8456451892852783, 2.12797212600708, 0.9537367224693298, -0.4554210305213928, 0.6494612693786621, -2.2814698219299316, 0.10653270035982132, -0.7598182559013367, -1.2856051921844482, -1.5828044414520264, 0.5251415967941284, 0.08930037170648575, 0.1597236692905426, 0.8423760533332825, -0.8460837602615356, -1.2355844974517822, -1.7763653993606567, -0.3351965844631195, 0.7601562142372131, -0.1494235247373581, 0.6411281824111938, -1.1810537576675415, -0.4691213369369507, -0.7659366726875305, -1.248377799987793], "dtype": "float32"}, {"__nd__": [1.8594735860824585, 0.8521407842636108, -0.7618005275726318, -0.7660171985626221, -0.057196855545043945, -1.4420539140701294, 0.4358271658420563, -1.294238805770874, -0.37043297290802, -1.0248225927352905, -0.47984829545021057, -1.3311572074890137, -1.5728219747543335, -0.966140866279602, -0.6800098419189453, -0.2948894202709198, 0.06898362934589386, 1.5268961191177368, 0.8122518062591553, -0.2528991997241974, 0.7259183526039124, -0.6888972520828247, 1.4484913349151611, 0.5494769811630249, -1.3558393716812134, 0.7553855180740356, 0.8160923719406128, -2.6855952739715576, -0.7480949759483337, -0.47382253408432007, -1.9285122156143188, -1.0671775341033936, -0.9483581781387329, 0.5070958137512207, 0.30046316981315613, 2.626173734664917, -0.39425939321517944, 1.1532310247421265, 0.07884647697210312, 0.2709257900714874, -1.1596298217773438, -1.9108997583389282, -0.7671481966972351, 0.7621685862541199, 1.5828056335449219, 0.6039910316467285, 0.49828091263771057, -0.838766872882843, 1.5342938899993896, 1.343806266784668, -0.5308324694633484, -0.046370670199394226, -0.9960519075393677, 1.4672068357467651, 0.406354159116745, -0.0950646698474884, 0.40689998865127563, 0.4406222999095917, -0.41149669885635376, -1.241565227508545, -1.2837454080581665, -1.7383664846420288, 0.46125301718711853, 2.2185418605804443, -1.6990692615509033, 0.13828745484352112, 0.41098394989967346, -0.9846256971359253, 0.20442579686641693, -1.0969369411468506, 1.2004021406173706, -0.5942490696907043, 0.10056038200855255, 0.6842415928840637, -0.42981937527656555, -1.7912307977676392, -0.5312556624412537, 0.7629952430725098, -0.08453265577554703, 0.22978918254375458, -1.0804017782211304, 0.14322291314601898, 0.7281669974327087, 0.03644802048802376, 1.9080145359039307, -0.20779357850551605, -1.041350245475769, -1.6176592111587524, 1.152005910873413, -0.05901824310421944, 1.0116218328475952, 0.4576689302921295, 1.6250842809677124, 1.2661073207855225, -0.3775545656681061, 0.2738848924636841, 1.085371971130371, -1.2360751628875732, 1.4508960247039795, 1.2119933366775513, 0.9829258918762207, -0.20093216001987457, 0.3021410405635834, -1.117239236831665, 0.057954173535108566, -0.587379515171051, 0.08656396716833115, -0.4946262538433075, 0.2838035225868225, 1.1793321371078491, -1.309603214263916, 0.925390899181366, -1.9918125867843628, -2.1610333919525146, -0.11936867982149124, 0.006729381624609232, 0.4428008496761322, -1.6473852396011353, -0.059938643127679825, 0.12652470171451569, 1.2552050352096558, -0.10383958369493484, 0.5289917588233948, -0.9774255752563477, -0.18755462765693665, -0.4455353319644928, 0.7033043503761292, -0.9888193011283875, -0.834322452545166, 0.5759881734848022, 1.7894366979599, 0.38270333409309387, 0.3035602867603302, -0.34614238142967224, 1.0884252786636353, 0.5399169325828552, 1.7331926822662354, 0.3307115137577057, -0.9486769437789917, -0.569941520690918, -0.9874045252799988, -1.2508208751678467, 1.2637174129486084, 0.8073369264602661, -1.2386443614959717, 1.7813206911087036, 0.24456073343753815, -0.9568920135498047, 0.3862219452857971, -0.0691942572593689, -0.7726547718048096, -1.942934274673462, 0.036719903349876404, 0.2919958531856537, -2.5790774822235107, 0.5549073219299316, 0.6176333427429199, -0.4697956144809723, 0.11212305724620819, -0.28065815567970276, -0.8371025919914246, -0.36005496978759766, 1.3516038656234741, -1.4506170749664307, 0.2371922880411148, 0.18411774933338165, -0.7076302766799927, -0.44033920764923096, -1.4004133939743042, -0.8519622087478638, 0.014951829798519611, 0.3267159163951874, -0.8983768224716187, 3.056342363357544, -0.5703445672988892, 0.1273300051689148, -0.23058007657527924, -0.3997797966003418, 1.301068902015686, -0.4788113832473755, 1.3343888521194458, 1.141816258430481, -1.0456651449203491, 1.0354934930801392, -0.6503781080245972, 2.7364587783813477, -0.3858303725719452, -0.9409262537956238, -1.911076307296753, -1.293045163154602, -0.5806488990783691, -0.9980278015136719, -1.2286869287490845, -0.6953397393226624, -0.20654773712158203, 1.4493473768234253, -1.7032948732376099, -0.5516409873962402, -0.9538936018943787, -1.5151478052139282, -1.266136646270752, -0.9057348370552063, 0.7170434594154358, -0.19390057027339935, -0.53798508644104, 0.6112917065620422, 0.2807694971561432, -0.2233288437128067, 0.3842858374118805, -0.8283509612083435, 0.6273813247680664, 0.4669249355792999, -0.4123042821884155, -1.2194762229919434, 1.520064353942871, 0.09420101344585419, 3.051265001296997, -0.6534090638160706, 0.8528429865837097, -0.4342191517353058, 0.25366804003715515, 0.8333911895751953, -0.7993078231811523, -0.10009046643972397, 0.16094160079956055, 0.1134801059961319, 0.5964510440826416, 1.02556312084198, 0.8336283564567566, 0.263012558221817, -0.34737294912338257, -1.1296836137771606, -0.7354984879493713, -0.5285487771034241, 0.68271404504776, -0.363610178232193, -1.24233877658844, 0.6263471245765686, -0.8032506704330444, -0.7658286094665527, -0.22627140581607819, 0.47317785024642944, -2.4131834506988525, -1.1214540004730225, 0.17558914422988892, -0.3641609847545624, -0.8645445108413696, -0.5414528846740723, 1.3852540254592896, -0.4557282626628876, 0.7126649618148804, 0.7918691039085388, -0.4502210021018982, 0.4434390664100647, -0.45023760199546814, -0.6104174852371216], "dtype": "float32"}], "e": {"__nd__": [-1.6704201698303223, -1.3748892545700073, 0.348736971616745, -1.675450086593628, 1.8569042682647705, 2.5862197875976562, -0.7612500190734863, 2.0680453777313232, 0.6516436338424683, 0.4709997773170471, 1.4574156999588013, 1.02060067653656, 1.2439980506896973, 0.17399412393569946, 1.1349679231643677, 0.19569137692451477, 0.47630512714385986, -2.1340818405151367, -0.6854239702224731, -0.639374852180481, 0.11554664373397827, 0.8769323229789734, -1.1179203987121582, -0.13897305727005005, 0.3450819253921509, 0.027795493602752686, 1.2406104803085327, 1.0471527576446533, -0.9813165068626404, -1.0310089588165283, 2.7699711322784424, 1.1958931684494019, 2.026700496673584, 0.21533507108688354, -0.08989135921001434, -2.3421356678009033, 0.2244988977909088, -0.28477078676223755, -1.2085623741149902, -0.6927846074104309, 1.4025686979293823, 3.712320566177368, 0.0026840567588806152, -1.84122896194458, -2.1460928916931152, 0.3652811646461487, -0.733286440372467, 2.163113832473755, -3.4068233966827393, -0.21528315544128418, 1.5656986236572266, -1.3722950220108032, 1.149633765220642, -0.25144922733306885, -0.3184351325035095, 1.0947661399841309, 1.967987060546875, -0.16669005155563354, 0.1311143934726715, 0.4705130457878113, 1.9318100214004517, 1.5416364669799805, -0.6399993896484375, -2.3238015174865723, 2.3489389419555664, -1.2046270370483398, -1.940860390663147, -1.4492510557174683, 0.9942450523376465, 1.1707303524017334, 0.3097172975540161, 0.5852927565574646, -0.8427157998085022, -0.20631229877471924, 0.35323095321655273, 0.5370441675186157, -0.3538109064102173, 1.0037840604782104, 0.4388832449913025, 0.18659736216068268, 0.8038500547409058, -0.8329424858093262, 0.16348880529403687, -0.14107447862625122, -2.6671226024627686, 0.07366505265235901, 0.13574421405792236, 1.807705044746399, -0.02277827262878418, -0.7770668864250183, 0.4169222116470337, -1.1252888441085815, -1.4716720581054688, -2.1024937629699707, 0.15531939268112183, -0.22648048400878906, -1.520056962966919, 0.5331872701644897, -2.128782272338867, -2.0331532955169678, -2.5530059337615967, -0.062037453055381775, 0.0990171730518341, 2.0256412029266357, 0.5891485214233398, 3.0447158813476562, 0.2321176528930664, 0.03829234838485718, 1.5878310203552246, -2.2265501022338867, 2.278080940246582, -1.88053560256958, 2.3459243774414062, 0.1926366090774536, 1.0186424255371094, -0.16497699916362762, -1.4104822874069214, 3.3258042335510254, 0.8252933621406555, -0.08071686327457428, -2.0006484985351562, 0.06047699227929115, -0.6929205060005188, 1.7022020816802979, 0.9856297969818115, -0.22222104668617249, -1.25267493724823, 0.4567539691925049, -0.514994740486145, -1.1675630807876587, -1.8818447589874268, 0.3082486093044281, 1.0168192386627197, -0.4622034728527069, -0.5374487638473511, -0.9843737483024597, 0.3419950008392334, -0.37943506240844727, 1.4508039951324463, -0.3662206530570984, 0.17671507596969604, 1.4525161981582642, -1.6477718353271484, -0.4540483355522156, -0.3231797218322754, -1.11931574344635, -1.1440410614013672, 2.6793155670166016, -0.6746891736984253, 1.166459321975708, -0.622328519821167, 2.371880531311035, -0.9247495532035828, -0.7788848876953125, 2.6036739349365234, -0.8719822764396667, -0.35337772965431213, 1.2174510955810547, 0.5406590700149536, 0.19317667186260223, -0.526327908039093, -0.4445911645889282, -1.5473600625991821, -0.6712088584899902, 0.5148882865905762, -0.423755943775177, 0.4563109874725342, 1.3887155055999756, 2.068323850631714, 1.0729632377624512, -1.0162854194641113, -0.01629433035850525, 1.2445544004440308, -3.941115617752075, 1.6136564016342163, 0.38055717945098877, -1.7390443086624146, 0.5217292308807373, -2.5657668113708496, 1.6151264905929565, -0.5273609161376953, -2.2454071044921875, 0.16079246997833252, -0.9336402416229248, 0.4115672707557678, -1.211255669593811, 1.126804232597351, 0.658499002456665, 2.4884722232818604, -0.7392340898513794, 0.8905135989189148, 1.864434838294983, 1.049734354019165, -0.014874458312988281, 0.7723811864852905, 0.5092673301696777, 1.9081637859344482, -0.5690367221832275, 2.399698495864868, -0.3633996248245239, 1.7693414688110352, 0.3237801790237427, 0.2135549783706665, -0.6555737853050232, 1.861832857131958, -0.31101861596107483, -2.0076100826263428, -1.7504842281341553, -0.700381338596344, 2.1225626468658447, -0.5003513097763062, -0.3998042345046997, 0.17640045285224915, 1.957376480102539, -1.142737627029419, 0.1861448734998703, -4.035987854003906, 1.8412623405456543, 0.5102373957633972, -0.8671344518661499, 2.0478270053863525, -0.11088460683822632, 0.3113018274307251, -1.8042653799057007, 0.495904803276062, 0.444031298160553, -1.1920828819274902, -2.8161163330078125, 0.29084843397140503, -0.8513314723968506, -0.12437212467193604, 3.9753289222717285, 2.8634705543518066, 1.482285499572754, -1.1381350755691528, 1.0130714178085327, -1.0391310453414917, -0.5198144316673279, 0.043432414531707764, -0.5197765827178955, -1.3565330505371094, 0.051963746547698975, 2.50248384475708, 1.2811776399612427, 0.6667869091033936, -0.48192277550697327, -0.3710399866104126, -1.2349125146865845, -1.7204506397247314, 1.2158844470977783, -0.8620885014533997, -0.15074092149734497, -0.7308327555656433, -0.9125604033470154, -0.3156990706920624, -0.6379603147506714], "dtype": "float32"}}, {"a": [{"__nd__": [-1.125994324684143, 0.8749562501907349, -0.39815932512283325, -0.5521423816680908, -1.127444863319397, 1.8088449239730835, 0.7418056130409241, -0.13280446827411652, -0.6947343945503235, -1.5297999382019043, 0.2918136715888977, 0.2884218096733093, -1.4084006547927856, 1.1615504026412964, 0.775956392288208, -0.9620737433433533, -0.5556965470314026, -0.4273206293582916, 0.16067896783351898, -0.14556942880153656, -1.6389391422271729, 1.474097490310669, 0.31631040573120117, 1.2607734203338623, 1.2548223733901978, -0.20790082216262817, 0.34196212887763977, 0.7366645932197571, 0.7696134448051453, -0.16391876339912415, -0.6181854605674744, 0.035286568105220795, -0.8015427589416504, -0.555933952331543, -2.030179262161255, 0.08364266902208328, -1.0268913507461548, -0.5558397173881531, -0.052700337022542953, 0.31383511424064636, 1.889171838760376, 0.20439636707305908, -1.4132649898529053, 0.1307464987039566, -0.5957258343696594, 0.3993854820728302, -0.6855278015136719, -0.7084068059921265, -0.5112637877464294, -0.6275627017021179, -1.8247809410095215, -0.6682717204093933, -0.014011094346642494, 1.200488805770874, -0.2931780219078064, -0.3717423677444458, 0.5798383951187134, 0.5394025444984436, 1.5995253324508667, 0.5851115584373474, -0.019450286403298378, 1.3383289575576782, 1.5294818878173828, -0.4753784239292145, 0.1581398844718933, -1.6806405782699585, -0.3641274869441986, -0.8837515711784363, -0.21407246589660645, -0.48163455724716187, -0.16604986786842346, -0.10303453356027603, 0.49346810579299927, 0.23093847930431366, -0.5618042945861816, -1.2891629934310913, 1.2348517179489136, 1.4173023700714111, -0.4544449746608734, 0.5614143013954163, -1.5261049270629883, -1.562214970588684, -0.26988452672958374, 1.4716817140579224, -1.0465514659881592, -0.8341242671012878, 1.6075947284698486, -1.8934478759765625, 0.6462222337722778, 0.48781391978263855, -0.20627076923847198, 0.18862496316432953, 2.621741533279419, -1.1268432140350342, -0.3139401972293854, -0.23812341690063477, -0.6896811127662659, -1.3418829441070557, 1.593300223350525, 0.22009506821632385, -1.757992148399353, -0.7028042674064636, 0.40540069341659546, 0.4034201204776764, -1.4403271675109863, -0.09142686426639557, 0.2608817219734192, 0.3925718367099762, 0.394599586725235, -1.823014259338379, 1.8469828367233276, 0.21113361418247223, 1.8059897422790527, 2.8630990982055664, -0.16055737435817719, -1.2337627410888672, 0.8641305565834045, -0.18083356320858002, -0.45140981674194336, -1.0332863330841064, 1.6877845525741577, -0.442674458026886, -1.3995847702026367, -0.44757139682769775, -1.1572588682174683, 0.15790416300296783, -0.4193645715713501, -1.0574241876602173, 0.6017661690711975, -0.6434339284896851, 0.723761260509491, 1.2609834671020508, -0.3619527816772461, -0.646514356136322, 0.19929607212543488, 0.8914495706558228, -0.002793061314150691, 0.13995584845542908, -0.026890793815255165, 0.3112146258354187, 0.6287429928779602, -0.706062912940979, -0.48253384232521057, 0.04358892887830734, -0.6657321453094482, 0.13612544536590576, 0.5361491441726685, -0.02172185853123665, -0.6035496592521667, -0.777202308177948, 1.751225471496582, 0.8785058259963989, -1.1199214458465576, -0.841026782989502, 1.8738447427749634, 1.5112977027893066, 0.12817631661891937, 1.1578489542007446, -1.4869812726974487, 0.001311955158598721, 0.7408092021942139, -1.8599673509597778, -0.7829942107200623, 0.7828548550605774, 1.1316298246383667, -0.5933074951171875, 0.2898094356060028, 1.600712537765503, -0.3819330930709839, -1.3333640098571777, -0.061407607048749924, -0.39315763115882874, 1.3010482788085938, 0.1601155549287796, -0.802094578742981, 2.4005446434020996, -0.5001170039176941, 0.4724457859992981, 0.6762315630912781, 0.7843220829963684, 0.17291589081287384, 0.5866738557815552, -0.06790213286876678, -1.4191819429397583, -0.2198106199502945, -0.19841206073760986, -0.17772026360034943, 0.13609173893928528, -0.6247028708457947, 1.3232574462890625, 1.566714882850647, -2.200361728668213, 0.009326023049652576, -0.7105945348739624, -2.606243848800659, 0.4101358950138092, -0.61879962682724, -0.3757345974445343, -0.6515779495239258, 1.0870054960250854, 1.2392981052398682, 0.2767953872680664, 0.693108856678009, -0.11242830008268356, 0.1401592344045639, -0.741762638092041, 0.03454280272126198, -0.7136704921722412, 0.3730262517929077, 0.8273311853408813, 0.31104275584220886, 1.1841291189193726, 2.6902453899383545, -2.1065611839294434, 0.013684019446372986, -0.5686805248260498, -0.45669546723365784, -1.3581751585006714, 0.05253523960709572, -0.2582727074623108, -0.6327707171440125, -0.2543226182460785, -0.5927437543869019, -2.3429219722747803, 1.0774089097976685, -0.3034726679325104, -0.9496373534202576, 0.30641207098960876, -0.6624383926391602, -2.173997402191162, 0.886902391910553, -1.6767091751098633, -0.21840709447860718, 0.12015624344348907, 1.349668264389038, 1.1900606155395508, 0.16574807465076447, -0.7633915543556213, -0.4158971607685089, 1.087701678276062, 1.7864493131637573, 0.5437886714935303, -0.6448445320129395, 1.0293720960617065, -0.3752400875091553, -0.40784934163093567, -0.2789686620235443, 0.5901489853858948, 0.2752748727798462, 0.9928014278411865, 1.2963932752609253, 0.5132455825805664, -2.332263469696045, -1.6965975761413574, 0.15858832001686096, -0.06470749527215958, 2.159196138381958], "dtype": "float32"}, {"__nd__": [-0.030214060097932816, -0.1526099294424057, 0.9438881278038025, 1.0644334554672241, -0.2354101538658142, 0.473202109336853, -1.1335467100143433, -0.9431517720222473, 1.233073115348816, -1.4071356058120728, 0.9223126173019409, 0.6778883934020996, 0.7697552442550659, 0.9128314852714539, 0.5638630390167236, -2.141458749771118, -1.0043795108795166, -0.6013116240501404, 0.9148204922676086, -1.127805233001709, -0.70489501953125, -0.3628120422363281, 1.3164317607879639, -1.441052794456482, -1.3539295196533203, 0.6271612048149109, -2.125527858734131, 0.3319779336452484, -0.45404091477394104, -1.5769513845443726, -0.4202713966369629, -1.0021369457244873, 0.16778475046157837, 0.3638220727443695, 1.4889615774154663, -0.012785236351191998, 0.10055350512266159, -0.5773108005523682, -0.023635121062397957, 1.0496025085449219, -1.925715684890747, 0.6591289043426514, -0.5351769328117371, 0.07389023154973984, 0.32504481077194214, 0.850594162940979, 0.23840324580669403, -0.39251017570495605, 0.31169435381889343, 0.01187314372509718, -0.20449985563755035, 0.20514768362045288, 0.8735805153846741, 0.1383320391178131, 0.15546925365924835, -1.2875103950500488, -0.127011239528656, -1.8580175638198853, 0.7200106382369995, -0.9392536878585815, 1.2953815460205078, 2.9934370517730713, -0.39739924669265747, 1.3296301364898682, 0.40359896421432495, -0.5569872856140137, -0.8413112759590149, 0.5619416832923889, 1.3948218822479248, 0.3537054657936096, -1.5118980407714844, -1.0422595739364624, -0.2279742956161499, 1.3571662902832031, 0.2834153175354004, 0.32726195454597473, 1.2105036973953247, -0.20459546148777008, -0.08911600708961487, -0.1818661093711853, -1.7779605388641357, 0.46667882800102234, 0.410838782787323, 0.9754424095153809, -1.1889032125473022, -0.8158632516860962, 0.9344695210456848, 0.32306161522865295, 1.375025749206543, -0.8630322217941284, 0.2008030116558075, -0.6759974360466003, 0.30975332856178284, -0.48504480719566345, 0.5359671115875244, 1.0813637971878052, 0.6752704977989197, -1.8082730770111084, 0.9404036402702332, 0.4475421607494354, -1.0430207252502441, -0.8857661485671997, 0.008306724019348621, 0.5735293626785278, -0.45070090889930725, -1.622511625289917, 0.28157907724380493, 0.16980864107608795, 2.4411327838897705, -1.2885593175888062, 2.5237505435943604, 1.7447023391723633, 0.3031228184700012, 1.6589840650558472, 0.09656038880348206, 0.9778372049331665, -0.7337658405303955, 0.07969794422388077, -1.0145952701568604, -1.5754709243774414, -0.27166053652763367, -0.6588938236236572, 0.5728123188018799, 1.2708039283752441, -0.16915297508239746, 0.05893224477767944, 0.18791940808296204, -1.6779682636260986, 0.2690810561180115, 0.011350513435900211, -0.046322934329509735, -1.1003749370574951, -1.9818800687789917, -0.2502603530883789, 0.3161567449569702, 1.3409130573272705, 1.1311759948730469, 1.455317735671997, 0.36995112895965576, 1.1561675071716309, -0.20601092278957367, 0.6284334063529968, 0.9318145513534546, 2.6893248558044434, 0.24263039231300354, -1.2725696563720703, 1.3777329921722412, -0.791299045085907, -0.06067719683051109, -0.29421472549438477, -0.9077565670013428, 1.3347725868225098, 0.596061110496521, -0.6120254397392273, 0.06933870166540146, -0.43323561549186707, 0.615195631980896, -0.37855708599090576, -0.7661634683609009, -0.6443074941635132, -0.43685364723205566, 0.2893671989440918, -0.9188750982284546, -0.08936968445777893, 0.4435810446739197, 2.337275266647339, 0.8562454581260681, -2.983765125274658, -0.511076033115387, -0.3969252407550812, 0.3203853368759155, 1.5693858861923218, 0.7420753240585327, -2.3029098510742188, 0.30697742104530334, 0.10806317627429962, 0.4129818379878998, 1.358868956565857, -1.0908515453338623, 1.1030701398849487, -1.653863787651062, 0.050822436809539795, 0.38059723377227783, 2.284532308578491, -0.702331006526947, -1.6939352750778198, 0.6871254444122314, -0.6612309217453003, 1.5638256072998047, -0.7080566883087158, -0.8772704005241394, 1.2444982528686523, 0.08543088287115097, 1.000783920288086, -1.4302150011062622, 1.651088833808899, -0.5889052748680115, 0.8194419741630554, -1.0049395561218262, -1.5506891012191772, -0.2568236291408539, 0.17446225881576538, 1.5766481161117554, 0.3500478267669678, 0.7155157327651978, 0.6589174270629883, 3.110154628753662, -0.8181433081626892, 1.255157232284546, -0.14925573766231537, 0.2607799470424652, -0.546752393245697, 1.1624075174331665, -1.2548614740371704, -1.3222780227661133, -0.20782722532749176, -1.158040165901184, 0.7032778263092041, 0.9218953847885132, -0.7529962658882141, 0.7838359475135803, -0.6620445251464844, -0.04423173889517784, 0.006761457305401564, 1.8764533996582031, -0.00042766297701746225, -0.9119616150856018, -0.9175471067428589, 1.1999969482421875, -1.6690126657485962, -0.4702436923980713, -0.3802131116390228, -2.4505131244659424, -0.46926018595695496, 1.0021568536758423, 0.23032237589359283, -0.9007649421691895, 0.6902670860290527, 0.5165442824363708, 2.0244996547698975, -0.742337703704834, -0.38872015476226807, 0.0410807728767395, -0.13868723809719086, 0.08637654036283493, -0.1074385941028595, -0.6948850750923157, -1.6044111251831055, -0.5609530806541443, -1.3925684690475464, -1.052015781402588, -0.54819655418396, 0.5027747750282288, 0.8477512001991272, -1.0444369316101074, 0.29739320278167725, 0.791949987411499], "dtype": "float32"}], "e": {"__nd__": [-1.0957802534103394, 1.0275661945343018, -1.3420474529266357, -1.616575837135315, -0.8920347094535828, 1.3356428146362305, 1.875352382659912, 0.810347318649292, -1.9278075695037842, -0.12266433238983154, -0.6304989457130432, -0.3894665837287903, -2.1781558990478516, 0.24871891736984253, 0.21209335327148438, 1.1793849468231201, 0.448682963848114, 0.17399099469184875, -0.7541415095329285, 0.9822357892990112, -0.9340441226959229, 1.836909532546997, -1.0001213550567627, 2.7018260955810547, 2.6087517738342285, -0.8350620269775391, 2.4674899578094482, 0.40468665957450867, 1.2236543893814087, 1.4130326509475708, -0.19791406393051147, 1.0374234914779663, -0.9693275094032288, -0.9197560548782349, -3.5191407203674316, 0.09642790257930756, -1.127444863319397, 0.021471083164215088, -0.029065215960144997, -0.7357673645019531, 3.814887523651123, -0.4547325372695923, -0.8780880570411682, 0.056856267154216766, -0.9207706451416016, -0.4512086808681488, -0.9239310622215271, -0.3158966302871704, -0.8229581117630005, -0.6394358277320862, -1.6202811002731323, -0.8734194040298462, -0.8875916004180908, 1.0621567964553833, -0.44864726066589355, 0.915768027305603, 0.7068496346473694, 2.3974201679229736, 0.8795146942138672, 1.5243651866912842, -1.3148318529129028, -1.655108094215393, 1.9268810749053955, -1.8050085306167603, -0.24545907974243164, -1.1236532926559448, 0.4771837890148163, -1.4456932544708252, -1.6088943481445312, -0.8353400230407715, 1.3458482027053833, 0.9392250180244446, 0.7214424014091492, -1.126227855682373, -0.845219612121582, -1.6164249181747437, 0.024348020553588867, 1.62189781665802, -0.36532896757125854, 0.7432804107666016, 0.25185561180114746, -2.0288937091827393, -0.6807233095169067, 0.4962393045425415, 0.14235174655914307, -0.01826101541519165, 0.6731252074241638, -2.2165095806121826, -0.7288035154342651, 1.3508461713790894, -0.4070737957954407, 0.8646224141120911, 2.311988115310669, -0.6417983770370483, -0.8499072790145874, -1.31948721408844, -1.3649516105651855, 0.46639013290405273, 0.6528965830802917, -0.22744709253311157, -0.7149714231491089, 0.18296188116073608, 0.3970939815044403, -0.17010924220085144, -0.9896262884140015, 1.5310847759246826, -0.020697355270385742, 0.22276319563388824, -2.0465331077575684, -0.5344549417495728, -0.6767677068710327, -1.5335687398910522, 1.5028669834136963, 1.2041150331497192, -0.25711774826049805, -2.211599826812744, 1.5978963375091553, -0.2605315148830414, 0.563185453414917, 0.542184591293335, 1.9594451189041138, 0.21621936559677124, -1.9723970890045166, -1.718375325202942, -0.9881058931350708, 0.09897191822528839, -0.6072839498519897, 0.6205440759658813, 0.33268511295318604, -0.6547844409942627, 0.7700842022895813, 2.361358404159546, 1.6199272871017456, -0.3962540030479431, -0.11686067283153534, -0.44946348667144775, -1.1339690685272217, -1.3153618574142456, -0.3968419134616852, -0.8449528813362122, 0.8347539305686951, -1.334496259689331, -1.4143483638763428, -2.6457359790802, -0.9083625078201294, 1.408695101737976, -0.8415838479995728, 0.7695772051811218, -0.5428724884986877, -0.48298758268356323, 2.658982038497925, -0.45626676082611084, -1.7159825563430786, -0.22900134325027466, 1.8045060634613037, 1.944533348083496, -0.48701930046081543, 1.5364060401916504, -0.7208178043365479, 0.6456194519996643, 1.1776628494262695, -2.14933443069458, 0.13588088750839233, 0.8722245693206787, 0.688048779964447, -2.9305827617645264, -0.5664360523223877, 4.584477424621582, 0.12914294004440308, -0.936438798904419, -0.38179293274879456, -1.9625434875488281, 0.558972954750061, 2.4630253314971924, -1.109071969985962, 2.2924814224243164, -0.9130988121032715, -0.8864231705665588, 1.7670831680297852, -0.3187480568885803, 1.8267797231674194, 0.5358514189720154, -0.4484993815422058, -3.703714370727539, 0.4825204014778137, 1.49552321434021, -0.8648456931114197, 0.7973226308822632, -2.188528537750244, 2.0313141345977783, 2.4439852237701416, -3.4448599815368652, -0.07610485702753067, -1.7113784551620483, -1.176028847694397, -1.240952968597412, -0.029894351959228516, -1.195176601409912, 0.3533616065979004, 2.6376945972442627, 1.4961217641830444, 0.10233312845230103, -0.8835392594337463, -0.46247613430023193, -0.5753564834594727, -1.4006800651550293, -3.0756118297576904, 0.104472815990448, -0.8821309804916382, 0.9765869379043579, 0.05026280879974365, 1.7308814525604248, 1.527837872505188, -0.851699709892273, 1.3359620571136475, -0.36085331439971924, 0.7013447284698486, -2.061452865600586, -0.8693601489067078, 0.4947235584259033, -1.4166066646575928, 0.4077219069004059, -0.5485120415687561, -2.3496835231781006, -0.7990444898605347, -0.30304500460624695, -0.03767573833465576, 1.22395920753479, -1.8624353408813477, -0.5049847364425659, 1.3571460247039795, -1.296496033668518, 2.2321059703826904, 0.5894164443016052, 0.3475114107131958, 0.9597382545471191, 1.0665130615234375, -1.4536585807800293, -0.9324414730072021, -0.9367979764938354, 2.528787136077881, 0.9325088262557983, -0.685925304889679, 1.1680593490600586, -0.4616166353225708, -0.30041074752807617, 0.41591641306877136, 2.1945600509643555, 0.8362279534339905, 2.3853697776794434, 2.3484091758728027, 1.0614421367645264, -2.835038185119629, -2.54434871673584, 1.203025221824646, -0.3621006906032562, 1.367246150970459], "dtype": "float32"}}, {"a": [{"__nd__": [-0.05768416449427605, -1.6552222967147827, -2.171543836593628, 0.7768170237541199, 0.8974946141242981, 0.25732946395874023, -1.0743112564086914, 1.4313939809799194, 1.1916710138320923, -0.06887307018041611, -0.2466956079006195, -1.9819658994674683, 0.7266573905944824, -1.6390289068222046, 0.6551365256309509, 0.7035932540893555, 0.06791538745164871, 1.5536949634552002, -0.34827232360839844, 0.563991129398346, 0.7713248133659363, 0.2232406586408615, -0.4440353214740753, 1.2050538063049316, 0.39755183458328247, -0.2177855223417282, -0.19103199243545532, 0.1052061915397644, 2.2559609413146973, 0.08128786087036133, -0.26410722732543945, -1.0897150039672852, -1.2446727752685547, -1.9226665496826172, 0.20931796729564667, 0.0030499424319714308, -1.0542199611663818, -1.5129491090774536, -0.3528880178928375, -0.5512251257896423, -1.3634700775146484, 0.2740352153778076, 0.11465111374855042, -0.07899660617113113, 0.743042528629303, -0.6816145777702332, 1.3307641744613647, -1.0441144704818726, 0.4848073422908783, 0.605466902256012, -0.29052314162254333, 0.32101044058799744, -1.3148454427719116, -1.5342612266540527, -0.23246248066425323, -0.761385977268219, 0.5603119730949402, -0.0012882982846349478, 0.5855029821395874, 0.28743281960487366, 0.17231988906860352, 0.06149497628211975, -0.40455344319343567, -0.4965536296367645, 2.0032663345336914, -1.7932113409042358, -0.21900758147239685, 0.971445620059967, 0.3867476284503937, 1.376178503036499, 0.5163381695747375, -1.2964743375778198, -0.3632681965827942, -1.4387403726577759, 0.9507840275764465, 1.4084936380386353, -0.4770096242427826, 0.5066640377044678, -0.4481562674045563, -1.4769049882888794, -0.4858768582344055, -1.62258780002594, -0.60924232006073, 0.1702650785446167, 0.6846247315406799, 1.2533355951309204, -0.3760601282119751, 0.19227996468544006, 0.37727537751197815, 0.5112543702125549, -2.896273374557495, -0.8240494132041931, 0.749706506729126, -1.1669541597366333, 0.19947242736816406, 0.7851218581199646, -1.9461561441421509, -0.2409152388572693, 0.18622881174087524, -0.04081464931368828, -1.582648754119873, 0.29349806904792786, -0.6152254343032837, 0.7675992250442505, 0.3742782175540924, -0.6071853637695312, 0.1174042671918869, -0.750373125076294, -0.7615083456039429, -1.6426588296890259, -0.3029184639453888, 0.7384740710258484, 0.0452035553753376, -0.0941203311085701, -0.7056782245635986, -0.6037241220474243, 0.662619411945343, -1.5775338411331177, 1.9779682159423828, -0.1505502164363861, 2.0843613147735596, 1.0848431587219238, -0.5098552703857422, 0.6407569050788879, 0.16089947521686554, 0.3645678162574768, 0.6942570209503174, -0.649107038974762, -0.7220830917358398, -0.5239389538764954, -1.4741833209991455, -0.3711738586425781, 0.11927618086338043, 0.1385548859834671, 0.07986626029014587, -0.5103360414505005, 0.6712254285812378, -2.1701290607452393, -0.5178883075714111, -0.4517649710178375, 0.7712400555610657, 0.5371501445770264, 0.6712082624435425, -0.11748121678829193, 0.8149265050888062, -1.1402305364608765, -0.06808546930551529, -2.2544384002685547, 1.5693837404251099, 1.4462136030197144, -0.7455211877822876, -0.9967437982559204, -2.247443914413452, -1.6819181442260742, -0.02870376594364643, -0.30808308720588684, -1.626361608505249, -0.6504944562911987, -0.8154922723770142, 0.31930768489837646, 0.5603264570236206, -0.3684084117412567, -0.03318636491894722, -0.15313653647899628, -2.2367160320281982, 0.03236013278365135, -0.21129076182842255, 0.6040788888931274, -1.9820560216903687, 0.6191354393959045, -0.7837515473365784, 0.12247712165117264, 1.067318320274353, -1.6195662021636963, -1.45499587059021, 0.8518490791320801, -1.283341646194458, -1.1015573740005493, 0.46058371663093567, -0.34484973549842834, 0.6372385025024414, 0.1593383550643921, -0.1660245656967163, 0.8451870083808899, 2.0494275093078613, -0.23850247263908386, 1.341495156288147, 0.9955576062202454, -0.02773718722164631, 0.3195452094078064, -1.0751593112945557, -0.7539150714874268, 0.1404169648885727, -0.18407370150089264, 1.1646713018417358, -1.43647301197052, -0.42969903349876404, -0.7740306258201599, -0.2880187928676605, 1.0286771059036255, -1.3421895503997803, 0.7153509855270386, -0.7247946262359619, -0.4585019052028656, 0.8650570511817932, -0.41671937704086304, 0.5347291827201843, -0.0181418526917696, -0.3049182593822479, -0.4918385148048401, -1.862717628479004, -0.8769486546516418, 0.0374499075114727, 0.07385676354169846, -1.6195133924484253, 0.8761609196662903, 0.4477158486843109, -1.4274100065231323, -1.1677672863006592, 0.3840368092060089, -1.883057713508606, -0.8650553226470947, -0.1745331585407257, -0.24704653024673462, -0.17608888447284698, 0.27312204241752625, -0.9158027172088623, -0.6034399271011353, 1.9392188787460327, -1.5175970792770386, -0.5632059574127197, -0.0016974463360384107, -0.758992075920105, 1.1117041110992432, -0.03721765801310539, 0.6493180394172668, -2.72525691986084, 1.0811818838119507, 1.04286789894104, -0.8627266883850098, 1.2021727561950684, 0.16170483827590942, 0.825612485408783, 1.7120083570480347, 0.0002495651424396783, -1.1130927801132202, -0.10945278406143188, -1.3741917610168457, -0.8989146947860718, -0.8598698377609253, -0.8060721158981323, 1.0740089416503906, 0.7450110912322998, -0.9882228970527649, 1.0429483652114868, -1.7688837051391602, -0.743859052658081, -0.22452428936958313, 1.1061038970947266, 0.6458882093429565, 0.3749275803565979, 1.2253470420837402, 0.6400579214096069, 1.2918273210525513, -0.861293613910675, -1.7644892930984497, -0.48962923884391785, 0.760435163974762, 0.19060951471328735, 0.8211829662322998, 0.642986536026001, 0.44874507188796997, 0.7522459626197815, -0.7828711867332458, 1.0279940366744995, -1.0725681781768799, -0.1798725426197052, -1.4815757274627686, 0.4410417973995209, -1.6183677911758423, 1.342259407043457, -1.4104642868041992, 0.1949695199728012, -0.20443454384803772, -0.9422008395195007, 1.1864378452301025, -0.4930852949619293, 1.403781771659851, -1.090940237045288, -0.8669387698173523, -2.065639019012451, 0.49004900455474854, 0.8544943928718567, 0.16178619861602783, -0.2248813807964325, -0.9848355054855347, 2.2904608249664307, 0.6814740896224976, -1.9084559679031372, -0.3724531829357147, -2.673896312713623, 1.5195969343185425, 1.9189127683639526, 1.4034454822540283, -0.11407287418842316, -0.6530067324638367, -1.4653700590133667, 0.4737057685852051, -0.8059431314468384, -0.43552613258361816, -0.6993252038955688, -0.36764630675315857, 0.8415088653564453, 1.6350250244140625, 0.09438502043485641, -0.03682010993361473, -1.171757698059082, -1.2878035306930542, -1.1213030815124512, 0.8323620557785034, 0.1940474659204483, 0.9175069332122803, 0.37222176790237427, -0.4556331932544708, -1.2989039421081543, -0.42109230160713196, -1.4847570657730103, 1.9582524299621582, 0.22039304673671722, -0.4184989929199219, 1.9530941247940063, -0.43463099002838135, 0.5106956362724304, -0.21289636194705963, 0.31127241253852844, -0.4179602861404419, 0.4484425187110901, 1.3871864080429077, -1.3169153928756714, -2.2828238010406494, 1.3138148784637451, 0.12269825488328934, 0.8338806629180908, -0.3374471068382263, 1.5105959177017212, 1.7908689975738525, -1.225091814994812, 0.6319369077682495, -0.662105917930603, -1.6730953454971313, -0.35406333208084106, 0.5225036144256592, -1.4346133470535278, -1.352382779121399, 1.1883809566497803, -0.026336198672652245, 0.34551072120666504, -1.2168606519699097, -0.725782036781311, 0.05475500598549843, 0.11864001303911209, 0.07292966544628143, 1.5303194522857666, 0.08749566972255707, -1.631136417388916, -0.9704214334487915, -0.1487247496843338, 1.1973499059677124, -0.7907614707946777, -0.49480342864990234, -0.12922054529190063, -0.8196932673454285, -0.9411965012550354, -0.3492226302623749, 0.8905210494995117, 0.3727596700191498, -0.40849289298057556, 0.2872982323169708, -1.4181429147720337, 0.8863456845283508, 0.6249907612800598, 0.17734120786190033, -1.7737293243408203, 1.9299778938293457, -3.082873821258545, -1.1358387470245361, -1.05674147605896, 0.5757168531417847, -1.6422926187515259, 0.11068214476108551, -0.28183940052986145, -0.4218744933605194, 0.7375885248184204, 1.3126822710037231, 0.27468377351760864, -0.40437835454940796, -2.4307315349578857, 1.3614318370819092, -0.42934057116508484, -1.9981865882873535, 0.037345025688409805, 1.3420746326446533, -0.2653687000274658, -1.30397629737854, 2.1610002517700195, 0.5023471117019653, 0.23221124708652496, 0.9130761027336121, -2.3094069957733154, -0.1832851618528366, -0.2782147228717804, 2.1836330890655518, -0.6794202327728271, -0.6668205261230469, -1.513960838317871, -0.9376837015151978, -0.31252431869506836, 0.16240283846855164, -0.30364057421684265, 1.928585410118103, 0.3973448574542999, 0.5460658073425293, 1.285568118095398, 0.5424215793609619, -0.6276331543922424, -1.6643781661987305, 0.26505324244499207, -0.9386478066444397, 1.2376688718795776, -0.8749958276748657, -2.3548190593719482, -0.05464346706867218, 0.9906517267227173, -0.3066585958003998, 0.08841252326965332, 0.05987963825464249, 1.6500389575958252, 0.20882609486579895, -0.13903889060020447, -0.722674548625946, -1.7640984058380127, 1.3299731016159058, 1.0666378736495972, -1.0667164325714111, 0.5185129046440125, 0.23180612921714783, -0.48669829964637756, -0.4182354211807251, 0.6644611358642578, -1.32455313205719, -0.6299918293952942, -0.17970962822437286, -0.10879043489694595, -0.3151904046535492, 0.11979362368583679, 0.43746417760849, -1.0086864233016968, -0.015181479975581169, -0.6372158527374268, 1.4293962717056274, -1.5287868976593018, -0.3395404815673828, 0.6087017059326172, -0.3898095190525055, -0.4382481276988983, -0.6462392210960388, 0.21919012069702148, -1.2790465354919434, -0.17945720255374908, -0.1986071765422821, -1.5798397064208984, 0.18472351133823395, -0.5638644099235535, 0.22395548224449158, 0.7853543162345886, 0.2625899910926819, -1.2826181650161743, 0.24391235411167145, -1.7035983800888062, -0.856326162815094, 1.6156924962997437, -0.0709863007068634, 1.5049612522125244, -0.25671783089637756, -0.06202363967895508, -1.2016505002975464, 0.8865357637405396, -0.2689480781555176, 0.5681955218315125, 0.37818244099617004, 0.48069408535957336, 0.27238282561302185, -0.09466559439897537, -0.054493241012096405, -0.7527654767036438, -2.2029943466186523, 0.448541522026062, 2.0646841526031494, 1.0532708168029785, 2.3286209106445312, 0.4577016830444336, -1.004087209701538, -1.0257338285446167, 0.7311682105064392, 0.32493311166763306, -0.1859883815050125, 0.7347155213356018, -0.9006407856941223, 0.5169571042060852, -0.29559090733528137, 0.15023237466812134, 1.1942921876907349, -0.14624015986919403, 1.3885489702224731, 1.3420274257659912, -0.7695750594139099, 2.3529815673828125, -1.2526766061782837, -0.09802676737308502, 0.7502437829971313, -2.304018020629883, 1.0520702600479126, 1.1597455739974976, -0.10576595366001129, 0.2847209572792053, 0.22781440615653992, -0.2014591246843338, 1.0961337089538574, -0.3666870892047882, -0.033628273755311966, 0.38144347071647644, 0.8726041913032532, 1.1285725831985474, -1.4871729612350464, -2.048772096633911, -0.12872198224067688, -0.04443484544754028, -1.0868302583694458, 0.3196191191673279, 0.6478272080421448, 0.3970660865306854, 1.682142972946167, -1.0410579442977905, 0.032284583896398544, 1.964917540550232, 0.6971940994262695, -1.1249806880950928, 0.21282939612865448, 1.2037330865859985, 2.1198809146881104, 0.6792243719100952, 1.1907978057861328, -1.120483160018921, -0.24655817449092865, 1.2169147729873657, -0.9690356254577637, -0.05874466150999069, -1.8914306163787842, 1.504004955291748, -0.1308390200138092, 0.5762900114059448, -1.523560881614685, -1.3311595916748047, 1.8426434993743896, -0.34697696566581726, -0.5516456961631775, 0.5191802382469177, 0.5625277757644653, -1.9903050661087036, -0.9897410869598389, 0.051460202783346176, 0.005332191940397024, 0.3565022647380829, -0.4356478750705719, -0.28431254625320435, 0.2181011289358139, -0.6991124749183655, -0.8472413420677185, -0.5568589568138123, 0.00551988510414958, -0.15307722985744476, -0.4866868257522583, -0.2565779387950897, -0.052685368806123734, 0.3217208981513977, 0.36372891068458557, -1.308255910873413, 0.9818397164344788, -1.6262742280960083, 0.5478275418281555, -0.7214757204055786, 0.5800697803497314, 1.0653151273727417, -1.911906123161316, 0.5997488498687744, 0.2668037414550781, -0.47981026768684387, 0.37745511531829834, -0.5120169520378113, 0.10783147811889648, -1.0740928649902344, 0.7793129086494446, 0.5820698142051697, 0.7364670038223267, -0.674790620803833, 1.8421326875686646, 1.9803335666656494, -0.10246419161558151, -0.4350287616252899, -0.42615175247192383, 0.6784842014312744, 1.0226951837539673, -0.7678739428520203, -2.059927463531494, 0.4434511661529541, -0.056522831320762634, 0.15756568312644958, 0.5055726766586304, -1.1373921632766724, 0.04788383096456528, -0.31847506761550903, 0.3603183627128601, 1.2102172374725342, -0.01994592323899269, -0.8869292736053467, 1.2320683002471924, 0.6894886493682861, -0.23200705647468567, -1.8462074995040894, -0.9132794737815857, 0.1127091646194458, 0.5185713768005371, -0.5670554637908936, 0.40290218591690063, -0.6771594285964966, -0.4651258885860443, 0.49618056416511536, -0.31388673186302185, 0.09869418293237686, 0.8383035063743591, 1.0935330390930176, -0.6380307078361511, -0.3001786470413208, -1.3799678087234497, -1.3606399297714233, 0.251238614320755, -0.5735312104225159, -1.9151813983917236, 0.48094886541366577, -0.8961246013641357, -0.3671487271785736, 0.3660072088241577, 0.3459245264530182, -0.11143821477890015, -2.0554535388946533, 0.513688862323761, -0.6518282294273376, -1.4771934747695923, -2.27124285697937, 1.2708759307861328, 0.08800659328699112, 0.05815843120217323, 0.5575367212295532, -0.13230621814727783, -1.5361459255218506, 0.6286789774894714, 0.9516841769218445, 0.46627458930015564, 1.4903634786605835, 1.8598579168319702, -0.8505296111106873, -0.4954051673412323, 0.8390634655952454, 0.12760484218597412, 0.16262097656726837, 0.8882540464401245, 0.693932831287384, -0.4903060793876648, 0.07781734317541122, -0.7599137425422668, -0.2419617921113968, -1.1594077348709106, -2.013274908065796, -0.5429627895355225, -0.6368897557258606, 0.8510331511497498, 1.5324455499649048, -0.016945701092481613, 0.43091514706611633, -0.6769775748252869, -0.9671727418899536, -0.3242279589176178, -0.3240925967693329, 0.25860458612442017, 0.07652775943279266, 1.6233525276184082, -0.2513558268547058, -1.685126781463623, 0.08731269091367722, -0.6926110982894897, 0.1722213178873062, 1.093596339225769, -0.6734892725944519, -0.10639679431915283, 0.48704251646995544, -0.9754443168640137, 0.1868683099746704, 0.5168662071228027, 0.7848259210586548, 1.0700829029083252, 0.5282309651374817, 0.36587098240852356, -1.4156368970870972, -2.702800750732422, -0.9879492521286011, 1.3991191387176514, 0.09468498080968857, -2.2741165161132812, -0.4422345459461212, 0.34088578820228577, -1.8695948123931885, 0.5536808967590332, -1.1642537117004395, 1.0231342315673828, 0.013324701227247715, 3.07791805267334, -1.345718264579773, 0.268210232257843, -1.4874389171600342, 1.1355645656585693, 0.11330019682645798, 0.9099231958389282, 0.09291716665029526, -1.1792811155319214, -0.6593710780143738, -1.2436387538909912, -1.0523303747177124, 0.13743898272514343, 1.0621953010559082, 0.8505674004554749, 0.5069001913070679, -0.5695209503173828, 0.5032516717910767, -0.8595872521400452, 0.19980043172836304, 0.830400824546814, -0.19202715158462524, -0.4742874503135681, -0.4371824264526367, 0.6066229343414307, -0.3147527277469635, -1.4238226413726807, -0.41875573992729187, -1.0670608282089233, 1.4905949831008911, 0.0067304628901183605, -1.4736065864562988, 0.2592809200286865, 0.7117609977722168, 0.37271347641944885, -0.3644985556602478, 0.14317163825035095, 0.2727260887622833, 0.99471116065979, 0.35749000310897827, 2.2550172805786133, -0.21901459991931915, 1.5186707973480225, 0.01337680034339428, -1.70320463180542, -1.1735271215438843, -1.8161441087722778, -1.716374397277832, 1.1934500932693481, -0.6597830057144165, 0.5962311625480652, -0.8914884924888611, -0.016077272593975067, -1.0528146028518677, 0.8584956526756287, -0.2565005421638489, -0.45375746488571167, -0.16609376668930054, 2.1632072925567627, 0.44248226284980774, 0.8287820219993591, -1.2402637004852295, -0.650181770324707, 0.2903788089752197, -1.853049635887146, 1.3266453742980957, 0.26607421040534973, -0.02117781713604927, -0.20103488862514496, -0.23728641867637634, -0.5066238045692444, 1.3822635412216187, 0.680540144443512, 1.2339898347854614, -0.3062189519405365, -0.8797315359115601, 1.3476430177688599, -1.2157063484191895, -0.610813558101654, 0.7041091322898865, 1.0755915641784668, -1.313665747642517, -0.27720141410827637, 0.02280375547707081, -0.13737456500530243, -0.8603001236915588, 0.6677958965301514, -0.5276079177856445, -0.24426107108592987, -2.2445075511932373, -0.11445223540067673, -1.937191367149353, 0.6976375579833984, 0.48515504598617554, -0.5116328001022339, 0.15552125871181488, 0.10260951519012451, -0.15232402086257935, 0.35446086525917053, -0.1286255568265915, -0.47975286841392517, -0.7957231998443604, 1.6605874300003052, 0.18704599142074585, -0.025989945977926254, 0.7125218510627747, -0.39242488145828247, -0.3861249089241028, 0.6560342907905579, 1.0165303945541382, 0.6614899039268494, -1.4792401790618896, 0.051307663321495056, -0.029162321239709854, 0.44514498114585876, -0.8853626847267151, -1.0486557483673096, -0.12242069095373154, -0.8892703056335449, -1.0645313262939453, -0.07595475018024445, 0.9115737080574036, 0.12720675766468048, 1.0997823476791382, -0.9956778287887573, 0.6933408975601196, -0.019559422507882118, -0.8788992762565613, -0.42367783188819885, -0.859511137008667, -0.22317193448543549, -0.7493523359298706, 1.2371691465377808, -0.3646109998226166, -1.4168174266815186, -0.2030312716960907, 0.37215232849121094, -0.3239510953426361, 0.8413278460502625, -0.9100593328475952, -0.8548151850700378, 1.595839262008667, -0.0054589021019637585, 0.5255355834960938, 0.4797893166542053, 0.560204267501831, 0.033116843551397324, -0.19746354222297668, 1.060309648513794, -0.4933304190635681, 0.10525012016296387, 0.6674454212188721, -0.398406982421875, -1.5200767517089844, 0.4448316693305969, 1.0563321113586426, 0.24358230829238892, -1.6578943729400635, 0.6238839030265808, 1.9301159381866455, -1.5274882316589355, -2.67283034324646, -0.2465474158525467, 0.07010246068239212, 0.7635635733604431, 0.8002099394798279, 0.08832968771457672, 0.5287854075431824, 0.32704517245292664, 0.11966052651405334, -0.0575559027493, -0.1516549289226532, 0.6882144212722778, 0.39634284377098083, -0.4195767045021057, 1.070731282234192, -0.7124344706535339, -0.22687944769859314, 0.13145461678504944, -1.861605167388916, -0.3697827458381653, 0.8361204266548157, 0.7959719300270081, -1.1674290895462036, -0.8108590841293335, -1.0519152879714966, -0.2304796278476715, -0.06270281225442886, -1.5572230815887451, 0.3104328513145447, 0.06987689435482025, -0.07141950726509094, -0.6565141081809998, -0.17939351499080658, -1.736404538154602, 0.22595466673374176, 0.20623694360256195, -0.10303641110658646, -1.1511954069137573, 1.2426573038101196, 1.1027604341506958, -1.8268890380859375, 0.4038737416267395, 0.1490987092256546, -2.503643751144409, 0.5051746368408203, 0.6347240805625916, 0.7031766176223755, 0.15780875086784363, -0.6346285939216614, -0.49817922711372375, 0.5001122951507568, 1.1430184841156006, -0.474871426820755, -0.2694265842437744, 0.6813260912895203, -2.0948126316070557, 0.4382167458534241, -0.22193937003612518, -2.1331627368927, 0.7538305521011353, -0.8869838118553162, 0.7748289108276367, 1.3443962335586548, -1.2076283693313599, -1.1618614196777344, -0.006180501077324152, -1.6981290578842163, 0.7161575555801392, 0.18854071199893951, 0.9625623822212219, -1.7751061916351318, 0.205144003033638, 0.46065908670425415, -1.7401152849197388, 0.8798456192016602, -1.3299658298492432, 0.18907971680164337, -1.2682000398635864, 0.7814422249794006, 1.6757750511169434, 0.07659681141376495, 0.050049446523189545, 1.287663459777832, -0.4125630557537079, 0.07393801957368851, 0.8257810473442078, 0.4423010051250458, 1.5669472217559814, 1.2048782110214233, 1.916924238204956, 0.030660003423690796, 1.145575761795044, -0.7384647727012634, 0.9466353058815002, 1.780165672302246, 2.365725040435791, -1.1437294483184814, -0.08740997314453125, 0.7784797549247742, -0.7718028426170349, 0.6224042773246765, 0.6154223084449768, -0.38469988107681274, -0.43352481722831726, -0.8095444440841675, 0.38982662558555603, 1.329747200012207, 0.37636598944664, 2.802685499191284, -1.3540253639221191, -0.47333845496177673, 0.750952422618866, -1.0229486227035522, 0.4340974688529968, 0.890478789806366, 0.268760621547699, -0.1259281188249588, 1.0140687227249146, 0.5026232004165649, 0.8031631112098694, 0.6287882328033447, -0.5633959174156189, -0.9839081168174744, -0.2779908776283264, 0.2677977681159973, -0.09359689056873322, 1.0193240642547607, 0.3415997624397278, 1.0412827730178833, 0.20527154207229614, -0.6158709526062012, 0.9154191017150879, 0.5760466456413269, -0.7390596866607666, -0.31708234548568726, 2.823262929916382, -3.2381246089935303, 1.1882244348526, 0.13449710607528687, 0.007733957841992378, -0.21984702348709106, -0.432588666677475, 2.394228219985962, -0.37002718448638916, -0.2360590547323227, 0.8500247597694397, 0.4364050030708313, -0.11936745047569275, -0.22624869644641876, -1.258602499961853, 1.1178666353225708, -0.9929436445236206, -2.3440375328063965, 1.0799472332000732, 0.784774899482727, 0.3788645565509796, -0.47352084517478943, 1.2149782180786133, -1.0324082374572754, -2.1288719177246094, 0.19373056292533875, 0.8295910954475403, 0.6458891034126282, 2.0866599082946777, 0.8543707728385925, -0.19642217457294464, -0.16269922256469727, 0.9108235836029053, -0.03422197699546814, -0.106646828353405, -1.7229634523391724, -0.28287258744239807, -1.2621136903762817, 0.25633421540260315, 0.1563849151134491, 0.7297999262809753, 1.7983875274658203, -0.156080424785614, 1.4558072090148926, -1.1015435457229614, 0.17785625159740448, 0.970085084438324, 1.9659745693206787, -1.074959635734558, -0.2339930385351181, 0.16173458099365234, -0.07942671328783035, -0.9293388724327087, -1.6063799858093262, -1.2007591724395752, -0.06090475991368294, -1.317184567451477, 1.7877393960952759, 1.2439781427383423, -1.8798930644989014, -0.47256362438201904, 1.3399122953414917, 1.4638315439224243, 1.1324249505996704, -1.4083058834075928, -0.3478362262248993, 1.293448567390442, 0.12080325186252594, -0.4372621774673462, 0.5206601023674011, 1.000207543373108, 0.15596792101860046, 0.26219475269317627, 1.0886526107788086, -0.5450602173805237, -0.895046591758728, 0.7803018689155579, -1.1800411939620972, 0.5117689967155457, -0.7130312323570251, 0.8537132740020752, -1.1725234985351562, 1.1483229398727417, 0.29878881573677063, -0.28950685262680054, -0.846695601940155, -0.14145256578922272, -0.9104341864585876, -1.184912085533142, 1.085803747177124, 0.6073000431060791, 1.2417933940887451, -1.340305209159851, -1.9352848529815674, -1.5732390880584717, 0.659714937210083, 0.8148453235626221, -1.8444082736968994, 1.0302156209945679, -0.4292910397052765, -1.1896929740905762, -0.0734301283955574, 0.5732715725898743, -0.21038536727428436, -0.17214487493038177, -0.8390958905220032, -1.4183493852615356, -0.3746272921562195, -1.6248241662979126, 0.6013286113739014, -0.8538427352905273, 1.310984492301941, -1.5897327661514282, 1.4056580066680908, -0.8849639892578125, -0.8937585949897766, 0.2923223674297333, 0.8996877074241638, -0.04304703697562218, -0.3604227602481842, 1.9949663877487183, 0.5067336559295654, -1.205919861793518, -0.2658815085887909, 1.6925562620162964, 0.32808783650398254, 0.8246201276779175, 2.034095525741577, -0.5974101424217224, 0.37286463379859924, -0.6701077818870544, -0.00796657893806696, 0.5362430810928345, -1.3380695581436157, 0.14383289217948914, 0.1402650773525238, 1.0536412000656128, -0.5967192053794861, -1.277974247932434, 0.3927540183067322, -1.1295641660690308, 0.17945660650730133, 0.5650867223739624, -0.883319616317749, -0.12728293240070343, -1.045332670211792, -2.8516995906829834, -1.4668793678283691, -2.102745294570923, 0.005956036038696766, -1.0837342739105225, -0.15684270858764648, -0.06248940899968147, 1.0516456365585327, -0.3625427186489105, -0.41752731800079346, 0.4915986955165863, 1.6041675806045532, -0.09169299155473709, 0.3277585804462433, -0.20709767937660217, -0.7166601419448853, -1.5608294010162354, -0.0943564847111702, -0.2827921211719513, -0.26283547282218933, 1.9753447771072388, -0.35028061270713806, 0.31866562366485596, -0.3619091808795929, 1.1712744235992432, -0.3021009862422943, 1.8542016744613647, -0.27398648858070374, 0.7683965563774109, 1.0113728046417236, -0.2469223141670227, 0.8478890657424927, 0.055139195173978806, -0.7693547606468201, -0.6817349791526794, 0.3254172205924988, 1.7820144891738892, -0.5003080368041992, 0.5382250547409058, -0.8163609504699707, 0.24909883737564087, -1.3157349824905396, -0.7981956005096436, -0.015190578997135162, -0.758568525314331, -0.27938205003738403, 1.003661870956421, -0.13330110907554626, -0.7273188233375549, -1.2448583841323853, -0.5408341288566589, 1.1470134258270264, -0.08782440423965454, -0.15282198786735535, 0.7169113755226135, 0.8330717086791992, 0.9999666213989258, -0.3654458224773407, -0.9238638877868652, -1.6781429052352905, -0.6933805346488953, -0.37907665967941284, 0.7238108515739441, 0.31339049339294434, -0.574506938457489, 0.8781892657279968, -1.2671295404434204, -1.0609914064407349, -1.1862655878067017, -1.101945161819458, -0.263658732175827, -0.8565833568572998, -0.7713770866394043, 1.329702377319336, -1.477421522140503, -1.0441076755523682, -0.2496286928653717, 0.4330914616584778, 0.4759969413280487, -0.27654609084129333, -0.37860459089279175, -0.2779085636138916, 0.7330363988876343, -0.9333489537239075, -2.3643388748168945, 0.17396438121795654, -1.181559443473816, -0.07051678746938705, 1.379034161567688, 0.08697081357240677, 1.8262510299682617, -1.6026766300201416, 0.22794164717197418, 0.37947529554367065, 0.8197693228721619, -0.7398414611816406, 0.15081894397735596, 0.11263386905193329, 0.08130290359258652, -1.3816522359848022, -1.7334989309310913, 0.5342570543289185, 0.34132900834083557, 0.032447002828121185, 0.6337178349494934, 1.0779047012329102, -1.1647974252700806, 0.7812798023223877, 1.4920605421066284, 1.025518536567688, 0.5961735844612122, 0.7125400900840759, -1.433439016342163, -0.484040230512619, 1.038877248764038, 1.4979459047317505, 0.4787657558917999, -0.435292512178421, -0.11598622798919678, 0.8775038719177246, 2.0465104579925537, 0.8197707533836365, -0.4900204837322235, 0.6583948731422424, 0.4700931906700134, 2.437310218811035, 0.6349297165870667, 0.07595491409301758, -0.03490279242396355, -1.1881831884384155, -1.0118175745010376, -0.4350179135799408, -0.7680857181549072, 0.23466965556144714, 0.14384062588214874, 0.985664427280426, 1.561429738998413, 0.20026618242263794, -0.1495281606912613, -1.8123613595962524, 0.42686110734939575, 0.7074688673019409, -0.7236720323562622, -0.3451692461967468, -0.6530508399009705, 1.1055718660354614, 0.6666060090065002, 1.2630127668380737, -1.160125970840454, 2.705773115158081, -0.8104183077812195, -0.9804724454879761, 0.23230203986167908, -0.5130236148834229, 1.8579856157302856, -0.37306416034698486, -0.6471200585365295, 1.0133109092712402, 1.2585958242416382, -0.803338885307312, -0.4437011778354645, -0.3267386853694916, -0.1288689821958542, 1.7425893545150757, 1.2885349988937378, 1.0963354110717773, 1.3034794330596924, 0.045401431620121, 0.899878978729248, -0.047030068933963776, -0.7273514270782471, 1.689764380455017, -1.7349340915679932, 0.02463855966925621, -0.5568152070045471, -1.645089030265808, -0.3646816313266754, 0.0833764597773552, 0.47454962134361267, -0.8132162690162659, 0.7647088766098022, 0.3173907697200775, 0.8198771476745605, -1.9184457063674927, -0.44616150856018066, -0.611343502998352, 0.5792811512947083, -0.736918032169342, -0.5103821158409119, -1.1428946256637573, -1.1956682205200195, -0.9371333718299866, 0.9603267908096313, -1.0678415298461914, -2.4419660568237305, 0.28170904517173767, -0.46347281336784363, 0.5408573746681213, 0.5220977067947388, -0.3267897963523865, -0.43486347794532776, -0.5680389404296875, -0.024507660418748856, -0.7937157154083252, 0.17864440381526947, 1.3651893138885498, -0.6828625798225403, -0.6441798210144043, -0.6523745656013489, 0.5960273146629333, 0.6401247382164001, 1.4182605743408203, 0.6987515091896057, 0.115028016269207, 0.44357308745384216, -1.3165191411972046, 0.5534001588821411, 0.16500310599803925, 2.0673346519470215, 1.3676997423171997, -0.8294602036476135, -0.5711075067520142, 0.8725170493125916, -2.069493293762207, -0.9215869307518005, 0.3651304543018341, 0.6393054723739624, 0.18133370578289032, 1.15939462184906, -0.2736698091030121, 0.09319069236516953, -0.19514697790145874, -1.093732476234436, 0.04364382103085518, -0.5523903369903564, 0.07865624129772186, -2.6122329235076904, 0.10782624781131744, -0.542829692363739, -0.6148855090141296, 0.044465385377407074, -0.47141727805137634, 0.9250152707099915, 1.245118498802185, 0.8815945386886597, 0.9631810188293457, 0.3633568584918976, 0.032206542789936066, -1.2790149450302124, -0.36725321412086487, 1.209609031677246, 0.765255331993103, -0.4873623549938202, 0.10976112633943558, 1.057189702987671, -0.15881840884685516, 1.6206908226013184, -0.7534146904945374, 0.7904736995697021, 0.6965380311012268, 0.175629124045372, 0.30450567603111267, 0.5415722131729126, -2.2606797218322754, 0.4736875295639038, 1.146658182144165, 0.40346139669418335, 1.1413828134536743, -1.1738649606704712, -0.062043070793151855, 0.33559852838516235, -0.25606274604797363, 1.1917484998703003, 0.4910253882408142, -0.50438392162323, 0.8196617960929871, 0.8219480514526367, -0.42955946922302246, 0.6312461495399475, 0.3827778398990631, 0.35923007130622864, -1.243722915649414, 1.2777721881866455, 0.6512500643730164, -0.020928483456373215, 0.35239723324775696, 0.3391672372817993, -0.2993408441543579, -0.5787031054496765, 0.10174089670181274, -0.5903832316398621, -0.4804534316062927, -1.1584869623184204, -0.2729315757751465, 1.0070338249206543, 0.21584738790988922, 0.0496722050011158, -0.3133956789970398, -0.4230882227420807, -1.4925446510314941, 1.6538082361221313, -0.7243353128433228, -1.4966055154800415, 0.4007449448108673, 0.2540442943572998, -0.5174185633659363, -1.4722031354904175, -1.3783347606658936, 1.5666409730911255, -0.11736693233251572, -0.6533749103546143, 1.8581122159957886, -0.5446667075157166, -0.9242268204689026, 0.5437135100364685, 0.3664811849594116, 1.5622435808181763, -1.5569605827331543, 0.2879199683666229, -1.6955201625823975, 0.4602357745170593, -0.6549345254898071, -0.46245938539505005, -0.7365298867225647, 0.8381124138832092, 1.6223056316375732, -0.08674325793981552, -0.2580324113368988, -0.5041547417640686, 0.5215559005737305, -1.0323855876922607, 3.542837142944336, -0.23457331955432892], "dtype": "float32"}, {"__nd__": [1.7110992670059204, 0.14646275341510773, -1.3749545812606812, -1.5659996271133423, -0.42559900879859924, -0.7226145267486572, -0.2529120147228241, -1.1586802005767822, 2.3419275283813477, 2.122375249862671, -1.8199559450149536, 0.47215914726257324, 1.1889452934265137, -0.3481675982475281, -0.2398485243320465, 1.0788718461990356, 0.5266706943511963, -1.231721043586731, 1.2115533351898193, -0.5065010190010071, 1.8446661233901978, -0.9802988171577454, -2.318434000015259, 0.07947167754173279, 1.1960036754608154, 0.4950469434261322, 0.13318660855293274, -0.950396716594696, 0.894843578338623, -1.8888808488845825, 0.7666478157043457, -1.2468217611312866, -0.6371145248413086, -1.362349271774292, -0.9584535956382751, 1.4544236660003662, -1.2285083532333374, 0.14590270817279816, -0.8543915748596191, 0.9852870106697083, -1.4200575351715088, -0.04898028448224068, 0.8578290343284607, 2.226093053817749, -1.250086784362793, -0.9419581890106201, -0.11043157428503036, 1.434007167816162, -1.0928786993026733, 0.766904890537262, -0.6961719393730164, -1.1117578744888306, -1.673313856124878, 0.06723839044570923, 0.9248359799385071, -0.17351976037025452, -2.2153031826019287, -0.6389637589454651, 0.9064785242080688, 0.8094131350517273, 0.20497989654541016, 1.4207957983016968, -1.3598716259002686, -1.3247967958450317, 2.3413403034210205, 0.6036403775215149, -0.28164398670196533, -0.5712156891822815, -2.6895627975463867, 0.7732498645782471, -2.1756842136383057, -2.0745410919189453, -0.12901069223880768, 1.451355218887329, -0.4805588722229004, 0.30142948031425476, -0.5203648805618286, 0.3245277404785156, -1.7716366052627563, 2.060295820236206, -1.0586341619491577, 0.4917670488357544, -2.158651351928711, 1.7291538715362549, 0.3658038377761841, 1.963981032371521, -0.7440488934516907, 0.983052134513855, -0.2516486644744873, -0.11652176827192307, 1.2535332441329956, -0.9893246293067932, 0.6854513883590698, 1.9161889553070068, 1.7193474769592285, 1.0218942165374756, 0.2768678665161133, -1.0439815521240234, 1.4077019691467285, -0.7473171949386597, 1.174185037612915, -0.3881271481513977, -0.0377960167825222, -0.05092935636639595, 0.5712065696716309, -0.49430251121520996, 0.13670162856578827, -0.5769766569137573, 0.4938935935497284, 0.24387793242931366, 0.39153167605400085, -0.9063240885734558, 0.28633707761764526, -0.0015732727479189634, 0.549441933631897, 0.007150738965719938, -1.0349926948547363, 0.6933623552322388, -0.7311891913414001, 0.24225512146949768, -2.157050371170044, 0.4634653627872467, 0.6248872876167297, -0.9851943254470825, 0.5445742011070251, -0.4286782145500183, 1.3552465438842773, 0.6253501772880554, -1.319269061088562, -0.2574752867221832, -1.8465760946273804, -0.37694013118743896, 0.5034579038619995, 0.7933986186981201, -0.15909013152122498, -0.2925484776496887, 0.11543546617031097, -0.6611009240150452, 0.2850038409233093, 1.0926997661590576, 0.1533597856760025, 0.8328916430473328, -0.8327222466468811, -0.5374012589454651, 0.3079197108745575, 2.021083354949951, 0.9972215890884399, 1.706825613975525, 0.024994133040308952, 1.2367045879364014, -1.7312662601470947, 0.5107945799827576, -2.136232376098633, 1.2344098091125488, 0.47536277770996094, 1.1728333234786987, 0.4006536602973938, -0.5111806392669678, -0.7583450078964233, 0.19034357368946075, 1.5179386138916016, 0.6362583637237549, -0.5476393699645996, -0.5741733312606812, -0.7140557765960693, -0.21285663545131683, -0.08962354063987732, -0.48599952459335327, 0.011497009545564651, -1.5937649011611938, 1.5872759819030762, 0.25513434410095215, 1.552268624305725, 0.3006666302680969, -0.18986518681049347, 0.09697723388671875, 0.23496727645397186, -0.65749192237854, -0.3709176778793335, 1.9477999210357666, 0.9649749398231506, -0.5084612965583801, 0.5460475087165833, -0.8278326988220215, 1.4385932683944702, -1.0005193948745728, 0.9254932403564453, 0.04932470992207527, 0.0583355613052845, -0.7963929772377014, -0.237268328666687, -0.746403694152832, -0.8476625680923462, 0.4902341961860657, 2.563768148422241, -0.32175326347351074, 1.0342055559158325, 0.011136399582028389, -0.6970954537391663, 1.378576397895813, 0.6656287312507629, 1.1750198602676392, 0.14082393050193787, 0.1719013899564743, -0.8003090023994446, -0.24413834512233734, 1.722747802734375, -2.2734012603759766, 0.4190336763858795, 1.5105336904525757, 0.835515558719635, 0.9181726574897766, -0.23669469356536865, 1.1709566116333008, 1.1989774703979492, -0.004878432955592871, 1.7235991954803467, -0.45494189858436584, -2.464566946029663, 0.024004487320780754, -0.07906690239906311, -0.8221701979637146, -0.42821821570396423, -0.5083826184272766, -1.8081122636795044, 1.27532958984375, 0.039555832743644714, 2.7391889095306396, 0.01905389130115509, 0.43321940302848816, -1.4868788719177246, 1.0574941635131836, -1.0638647079467773, 0.5753738284111023, 0.9638206362724304, 0.8013158440589905, -0.8060057759284973, -0.7756808996200562, 0.7937234044075012, 1.2230476140975952, 1.4872123003005981, 0.0026407933328300714, -0.07533128559589386, 1.8349624872207642, -0.5658937692642212, 0.7948793768882751, -0.2850787937641144, -1.4035505056381226, -1.095445156097412, 0.4824409782886505, 0.24858605861663818, 0.5219204425811768, -0.3272359073162079, -0.47922438383102417, 0.3108484447002411, -0.15996360778808594, -1.5903748273849487, -0.4139491021633148, 1.726825475692749, -0.7457051873207092, 0.5741087794303894, -0.061414849013090134, 0.7546809911727905, -0.8562801480293274, -1.2661199569702148, 0.4654383361339569, -0.6899311542510986, -0.022290203720331192, -0.30257150530815125, 0.9801048040390015, 0.2696744501590729, 0.15310189127922058, -0.9661700129508972, -1.5635559558868408, -0.8302085399627686, 0.8468899130821228, 0.7036922574043274, 0.594161331653595, -0.39450597763061523, -1.0200659036636353, 1.1589710712432861, 0.19003847241401672, 0.5421959757804871, -0.9379316568374634, -0.6806732416152954, 0.6513448357582092, 0.7443249821662903, -0.9949336051940918, 0.2927456498146057, -1.1676075458526611, 1.2238842248916626, 0.6671788692474365, 2.2782130241394043, 1.513382077217102, -0.6573042273521423, 0.199662446975708, -0.4736765921115875, 0.1434059590101242, 0.7844278812408447, -0.004554013721644878, 0.061959292739629745, -1.3365018367767334, 0.5796160697937012, -0.013706262223422527, 0.6252310872077942, 1.0407074689865112, 0.0328691303730011, 0.6988276243209839, -0.6028502583503723, 0.9446959495544434, -0.0218929722905159, -0.5910807847976685, -0.2302023023366928, -1.4166874885559082, -0.6485128998756409, -0.42892277240753174, 0.02499537542462349, -0.08815723657608032, -0.6439706683158875, -0.18654541671276093, -2.1340577602386475, 0.08740939199924469, 0.5519008040428162, 0.7813260555267334, 1.0684963464736938, -0.2108563929796219, -0.09002663195133209, -0.48357415199279785, -0.48872148990631104, 0.11455534398555756, 1.3282140493392944, -1.6929341554641724, -0.5047447085380554, 0.2632422149181366, -1.0053799152374268, 0.39823347330093384, -1.3795140981674194, -0.36154696345329285, -1.6605346202850342, 0.254129558801651, -0.8009020686149597, 0.6213808059692383, -0.3146427869796753, -0.9777631759643555, -0.4430396854877472, 0.09733984619379044, -0.04106021672487259, -0.2849807143211365, -0.3790735602378845, -0.022174064069986343, 0.8050410151481628, -0.6009753346443176, 0.4012320041656494, -1.3490517139434814, 0.8816702961921692, 0.32594960927963257, -0.3816809356212616, -0.3595748841762543, -1.7144432067871094, 0.05828336626291275, -0.2757657468318939, -1.4157593250274658, -0.4704374372959137, 0.5383195281028748, 0.14213740825653076, 0.23802253603935242, -0.5809425711631775, 0.8086715340614319, -0.05388939380645752, 0.8114882707595825, 2.165269613265991, -2.0097556114196777, -0.5245217084884644, -0.019236121326684952, -0.7534506320953369, 0.9982627034187317, -0.3683472275733948, 1.634285807609558, 0.7298340797424316, 0.4006398022174835, -0.0368427038192749, 0.8208796381950378, 0.4727008044719696, 1.4428423643112183, 0.40567371249198914, 1.0134533643722534, 0.39780473709106445, 0.3267172574996948, -1.5443369150161743, 0.007478527724742889, 0.3877391815185547, -1.15252685546875, -1.371173620223999, -0.30035191774368286, 0.29230406880378723, -1.6815730333328247, 0.3103877902030945, 0.9500108361244202, -0.24518154561519623, 0.2391921728849411, 0.46176087856292725, -0.6527953147888184, 0.6158219575881958, 0.4475945234298706, -1.224319577217102, 0.5139133930206299, 0.8678262829780579, -1.065469741821289, 0.8610355257987976, -0.26137199997901917, -1.3266267776489258, -1.0259321928024292, -0.8750951886177063, 0.5351508259773254, -0.47602200508117676, -0.3882509171962738, -1.2520923614501953, 0.021674655377864838, -1.2331069707870483, -0.6652675271034241, 0.6062825918197632, 0.6581602692604065, -0.5266315937042236, -0.05116594210267067, 0.5265330672264099, -0.010682877153158188, 1.1934698820114136, 1.328218698501587, -0.08616212010383606, -1.2196974754333496, -0.1514236032962799, 1.3368836641311646, 1.1188526153564453, 1.175728440284729, 1.1101353168487549, -0.5069398283958435, 0.2765728235244751, 1.0804085731506348, 0.26806700229644775, 0.8783236145973206, -0.9938022494316101, -1.202974557876587, 2.005671262741089, 0.7718299627304077, 0.6161792278289795, 1.466430425643921, 2.2246346473693848, 2.875903606414795, -2.3774850368499756, 0.40034234523773193, 1.0439486503601074, -1.8920451402664185, -1.764421820640564, -0.16061583161354065, 1.899247407913208, 0.9551530480384827, 2.1677443981170654, 0.304002970457077, 1.089057445526123, 1.5382544994354248, 0.4765867590904236, -0.001579313655383885, 0.13953565061092377, -3.6721343994140625, 0.33569902181625366, -1.8496617078781128, -0.8077492713928223, 0.7156739234924316, 0.40663573145866394, -1.9930164813995361, 0.5328704714775085, 0.7573257684707642, 0.2033633589744568, 0.6917820572853088, 0.5719380378723145, -1.704042911529541, -0.20989197492599487, -0.2988009750843048, -1.22044038772583, -0.052909549325704575, -0.8221944570541382, -0.0897228941321373, -0.8727836012840271, 1.040949821472168, 2.2107126712799072, -0.5818920731544495, 0.03125583752989769, 0.04364084079861641, 0.43561458587646484, -0.18332809209823608, -2.242994546890259, 0.34122633934020996, -0.14270581305027008, -1.0899779796600342, -0.21620996296405792, -0.524953305721283, 0.2188526690006256, -0.3381167948246002, -0.30064669251441956, 0.08640047162771225, 0.22475023567676544, -0.005634732078760862, 1.1521656513214111, 0.22148919105529785, -0.7927525639533997, 1.709337592124939, 0.09788137674331665, 1.5465000867843628, -1.0878711938858032, 0.5423288941383362, 1.7030184268951416, 1.252334475517273, 0.07038760930299759, -0.608680784702301, -0.6271748542785645, -0.20769506692886353, -0.9955698847770691, -0.7706207036972046, -1.0199687480926514, -1.0251208543777466, 0.21506436169147491, -1.7940033674240112, 0.14077472686767578, 0.6660707592964172, -0.9683928489685059, -0.9367112517356873, -0.0011979315895587206, -0.06648784875869751, 0.9284476041793823, -2.0197834968566895, -0.09293732047080994, 1.3825937509536743, 0.06189466640353203, 1.7504079341888428, 2.164891242980957, -0.5697734951972961, 1.4816160202026367, 0.5003448724746704, -0.03255581483244896, -0.2995128035545349, -0.2392590492963791, -0.931026041507721, -0.18752622604370117, 0.9236136674880981, 1.2458808422088623, -0.39868542551994324, 0.5158209204673767, 0.4635680913925171, -2.3801941871643066, 0.6997434496879578, -1.2510634660720825, -1.1645587682724, 0.8264270424842834, 1.6650280952453613, -1.1292119026184082, -0.9035840630531311, -0.9696030616760254, -0.5485959053039551, -0.4044066369533539, -0.8528173565864563, -1.6550410985946655, 1.7518467903137207, -0.525192141532898, -0.7882779836654663, 0.8765859007835388, -0.3925667107105255, 1.061131238937378, 0.7607631683349609, 0.9631982445716858, -0.4257277548313141, 0.6120988726615906, -1.579738736152649, -1.401758074760437, 0.043377071619033813, -0.3309718668460846, 1.324105143547058, -0.2953600287437439, -1.7945830821990967, 0.04039802402257919, -0.3162481188774109, 0.16984739899635315, -1.1878447532653809, -0.7975211143493652, 0.04305914416909218, 0.723475992679596, 0.3941555321216583, 0.7366488575935364, -1.4725348949432373, -0.23013898730278015, 0.20433928072452545, 0.9529507756233215, 0.25303706526756287, -0.25247102975845337, -0.13024292886257172, -0.09898679703474045, -1.4389554262161255, 0.02135510928928852, 0.8001600503921509, 0.7059174180030823, -1.8700460195541382, -0.005767036695033312, -1.5101478099822998, 1.0904793739318848, 0.5361027121543884, 0.7056496143341064, -0.6704426407814026, -1.9399821758270264, -1.0480941534042358, -0.3562680184841156, -1.5038546323776245, -1.0102578401565552, 1.752161979675293, -1.3220951557159424, 0.5459501147270203, -1.8676270246505737, 0.24978891015052795, 0.8358497023582458, 0.44511035084724426, 1.2042109966278076, 1.7960398197174072, -1.3807812929153442, 0.2657265365123749, 0.5228245258331299, 0.42947855591773987, 1.5973403453826904, 0.28690439462661743, 1.2158865928649902, 1.1721991300582886, 0.16506248712539673, 0.5671490430831909, -0.7443009614944458, 0.33739668130874634, 0.21499748528003693, -0.122414730489254, -0.43399202823638916, 0.9998359084129333, -0.07756499201059341, 2.3123939037323, 1.3446712493896484, 0.7978967428207397, -0.5412389039993286, 0.9473435878753662, 0.1839648336172104, 0.1320328414440155, -1.0494771003723145, 0.6277533769607544, 0.29713454842567444, -0.413149356842041, -0.6205340027809143, -0.2202683985233307, -1.26777982711792, -0.4958045184612274, 1.4350035190582275, -0.5917496681213379, 2.3670828342437744, -1.28068208694458, 1.1471902132034302, 1.142344355583191, 1.2195842266082764, 0.630556583404541, -1.546189785003662, 1.203586220741272, 1.1261482238769531, 0.09853468835353851, -2.300971508026123, 1.1288268566131592, -0.8700227737426758, -0.2205706685781479, 1.2941343784332275, 1.3811248540878296, 0.4376131594181061, -0.5035259127616882, 1.7578742504119873, 1.2057433128356934, -0.10234472900629044, -0.912492036819458, 0.33467912673950195, -1.3073054552078247, 1.8157321214675903, 0.5510655045509338, -0.31471318006515503, 0.44976088404655457, -1.6332162618637085, -0.030587607994675636, -0.6056435704231262, -1.5508911609649658, -0.6033419370651245, 0.37847569584846497, -1.4050829410552979, -0.1702469289302826, -1.434867024421692, 0.22444970905780792, -1.7433741092681885, 0.7362951040267944, 0.6851625442504883, 0.8525142073631287, 1.829732060432434, 0.1888849288225174, 0.13408654928207397, 0.6200286149978638, -0.4545169472694397, -1.8921653032302856, -1.218280553817749, -1.216025710105896, 0.2026216685771942, -0.6331133246421814, -0.3893851935863495, 0.5712274312973022, 0.19463102519512177, 2.0121724605560303, -0.0449087955057621, -0.939849853515625, 1.2769962549209595, -0.41032564640045166, -0.06572072952985764, 1.004217267036438, 0.1882103979587555, 0.7089825868606567, -0.6114867329597473, -0.28309696912765503, 0.6584069132804871, -0.5175263285636902, -0.37361520528793335, -0.01417987048625946, 1.6270250082015991, 1.5118173360824585, 0.2692309319972992, 0.1096891313791275, -0.6558306813240051, 1.373085856437683, 0.46459442377090454, 1.0013960599899292, -1.3521941900253296, 0.8680069446563721, 0.21844206750392914, 0.8258865475654602, 1.109555721282959, 0.08454126119613647, -0.2914345860481262, 0.7707958817481995, -1.130379557609558, 1.5127414464950562, 0.429996132850647, -1.0283288955688477, 1.3811606168746948, 1.0581575632095337, 1.6330825090408325, 1.1034338474273682, 0.5978856086730957, 0.06662014871835709, -0.9531407356262207, 0.8657772541046143, 0.7064789533615112, -0.30853772163391113, 0.7099415063858032, -0.7771679162979126, 1.1790016889572144, 0.27081313729286194, 0.36362940073013306, -0.6123325228691101, 1.065495252609253, 0.27917957305908203, -0.4599696397781372, 0.5015937685966492, -0.09571631252765656, 1.155182957649231, -1.4118762016296387, -1.0424143075942993, 2.132988214492798, -0.3905634582042694, 0.5996376276016235, -0.30103033781051636, 0.3152191638946533, 2.7249114513397217, -1.8804370164871216, 0.6440819501876831, 0.15342776477336884, 0.13004963099956512, -0.6950130462646484, -0.8587899208068848, 1.4585832357406616, 0.6006597876548767, -1.109179973602295, 0.09446866065263748, -1.2484240531921387, 2.966803789138794, 0.46631839871406555, -1.7195465564727783, 1.013907551765442, 0.7124332785606384, 0.39682576060295105, 0.26275205612182617, -0.4709456264972687, -0.10998918116092682, -1.946672797203064, 1.3878579139709473, 2.0904953479766846, 0.5801870226860046, 0.6946645975112915, -1.29137122631073, -0.3525952994823456, -1.3465156555175781, -1.238564372062683, 0.6135305762290955, 1.4941328763961792, 0.004918014630675316, 2.5590224266052246, 0.8973681330680847, -2.1947619915008545, -0.3375425934791565, 1.3761121034622192, 0.22490231692790985, 1.1084762811660767, -1.6250889301300049, 1.3457093238830566, -1.1246594190597534, 0.4059887230396271, 2.1903231143951416, 0.1363711953163147, -0.17238497734069824, 0.8267885446548462, -0.7753438949584961, -0.8961399793624878, 1.0395305156707764, -1.4406442642211914, -1.1912215948104858, 1.8219884634017944, -1.2011158466339111, 0.4994199275970459, 1.1651691198349, -0.006021819077432156, -1.0146315097808838, 0.8779134750366211, -0.8878394961357117, 2.1743409633636475, 2.2756264209747314, -0.9827957153320312, 0.6320819854736328, -0.6598246097564697, 0.42769667506217957, -0.20835909247398376, -0.9395201802253723, 0.5623135566711426, -1.614785075187683, 0.3548608720302582, -0.11476288735866547, -0.7454066276550293, 0.9469011425971985, 0.07999682426452637, -0.3110736012458801, -0.12963292002677917, -0.18954890966415405, 0.9715766906738281, -0.25733476877212524, -2.8776135444641113, -3.0883285999298096, -1.0276843309402466, -1.233116865158081, -0.24562443792819977, -1.333963394165039, -0.7816522121429443, 1.3546051979064941, -1.3139445781707764, 0.8804639577865601, -0.09372976422309875, 1.3378208875656128, 0.9328727126121521, 0.09585600346326828, 1.123231291770935, 0.2664634883403778, -1.0843701362609863, 0.5453708171844482, 0.5555820465087891, -0.06285228580236435, 0.3163452446460724, -0.3763779401779175, 0.3885728418827057, -0.21212603151798248, -1.483911395072937, -0.20273053646087646, 1.6659702062606812, 0.8972136378288269, 1.5765180587768555, 0.07003401964902878, -0.3394787907600403, -0.9998963475227356, 0.22074945271015167, 1.0936810970306396, 0.8886599540710449, 0.812297523021698, -0.44212013483047485, 0.1765066236257553, -0.708922266960144, 0.8975095748901367, 0.6689116358757019, -0.5723939538002014, 0.19328153133392334, -2.2074239253997803, 0.7712076902389526, -0.7577100396156311, -0.30684757232666016, 0.6680749654769897, 0.7394810914993286, -0.6139298677444458, 1.656639575958252, 2.0862321853637695, 1.5132452249526978, 1.3750227689743042, 0.11683332175016403, 1.4558509588241577, -0.8212445974349976, -1.0551930665969849, 0.2217378169298172, 0.3158217668533325, -0.8074706196784973, 1.8443613052368164, -1.409294843673706, -2.0176172256469727, -0.34821265935897827, 1.6991658210754395, 0.804936408996582, 0.40185222029685974, 0.19088445603847504, 0.6161001324653625, 0.3726317882537842, 0.1503944844007492, -0.07462257146835327, -1.5615195035934448, 1.019322156906128, -0.38484013080596924, 0.423489511013031, 0.5379770398139954, 1.0831366777420044, -0.7486082911491394, 0.8692666292190552, 0.8964704871177673, -0.06674063950777054, -0.009269509464502335, 0.19077163934707642, -0.9856916069984436, -0.4170205891132355, -1.09992253780365, 1.2292917966842651, -0.745827317237854, -1.0379440784454346, 0.190830260515213, 0.5742000937461853, -0.33579981327056885, -0.9204462766647339, 0.4531204104423523, 0.8846974968910217, -1.609741449356079, 0.9389282464981079, -1.7481410503387451, 0.6896402835845947, -0.4120578467845917, -0.2248103767633438, 0.748386025428772, 1.4737955331802368, 2.1195642948150635, 1.4005024433135986, 0.7171677350997925, -1.3109636306762695, -0.3774491548538208, 0.16090594232082367, -0.5717507004737854, 0.6129024028778076, -0.21625249087810516, 1.8208919763565063, -0.15674838423728943, -1.2424181699752808, -0.486751914024353, -0.6936129927635193, 0.7350521087646484, 1.4271827936172485, 1.047385334968567, 1.0513876676559448, -0.6598599553108215, -1.0781614780426025, -0.7740970849990845, 0.06413854658603668, -0.31282761693000793, 0.3197999596595764, 1.0607112646102905, 0.11946354806423187, 1.7449816465377808, -0.7340477108955383, 0.6775440573692322, -0.015664584934711456, 1.0530520677566528, 0.011592460796236992, -0.48711591958999634, -0.08064472675323486, -0.8367133140563965, -1.0558781623840332, 0.8441870808601379, -0.562627375125885, 1.2548635005950928, 2.555541753768921, -0.3373451828956604, 0.9110439419746399, -0.04298315569758415, -1.859595537185669, -0.060142360627651215, 1.5130177736282349, -1.1845645904541016, -0.1351901739835739, -0.14801527559757233, -1.423974871635437, 0.34164875745773315, -0.12309049814939499, 0.6028298139572144, 0.07355956733226776, -1.3971474170684814, 0.15218134224414825, 0.6456837058067322, -0.23359882831573486, 0.5781697630882263, 0.5613329410552979, -2.144343376159668, 0.6357173919677734, -2.0225841999053955, -1.2910290956497192, -0.1372600644826889, -0.9880204796791077, -0.8198509812355042, 1.097625494003296, -1.091219425201416, 1.167759895324707, 1.8097245693206787, 1.3842496871948242, -1.1301679611206055, -1.5638151168823242, 0.03382745385169983, 2.095151901245117, -0.24441580474376678, -0.5950755476951599, -0.15514332056045532, 0.7216176986694336, 1.1625745296478271, -0.03619862347841263, -0.16491517424583435, 0.47653698921203613, 0.0008608899661339819, -0.44532346725463867, -1.8039604425430298, 0.08663608133792877, -0.6388055086135864, -0.2693180739879608, -0.32979440689086914, -0.9203943610191345, 0.5222203135490417, 1.036702275276184, 0.4960609972476959, -0.6873085498809814, 0.1383221596479416, 0.6669526100158691, -0.13615825772285461, 0.6662148833274841, -1.4491643905639648, -0.25030186772346497, 1.4065227508544922, 0.9669644236564636, -0.15726685523986816, 0.0934765487909317, 2.1286728382110596, 1.068759799003601, -0.7270112037658691, 0.28818294405937195, -0.4779111444950104, -0.6106978058815002, -1.2281749248504639, 0.6979067325592041, 2.208270311355591, 0.8473517894744873, -1.3194340467453003, 0.520145058631897, -1.4704229831695557, 0.21690501272678375, 1.047805905342102, 1.4756548404693604, 0.1191931813955307, 0.3207665681838989, 0.011584118939936161, -0.17104724049568176, 1.5632784366607666, -0.9521116614341736, -1.5923696756362915, 1.7535455226898193, 2.2677412033081055, -1.5938146114349365, 0.939160943031311, 0.8087080717086792, -0.4185408651828766, -0.9535808563232422, 0.16009044647216797, -0.3313848376274109, 0.8210670351982117, -0.6302701234817505, 1.1823337078094482, -0.6285284161567688, 1.6330482959747314, 0.8917083144187927, 1.1413123607635498, -0.3828856348991394, -0.22618556022644043, -0.08869412541389465, 1.188032865524292, -0.658076822757721, -0.7806623578071594, -0.22228838503360748, 0.3595961928367615, -0.07660584151744843, -0.0609104260802269, -1.310854196548462, -0.6156280636787415, -1.0697605609893799, 1.049607515335083, -0.039714694023132324, 0.6801607608795166, -1.3130460977554321, -0.03619619831442833, 0.37487736344337463, -1.0548303127288818, -0.645553469657898, -0.5165618658065796, 0.14733338356018066, 0.28131696581840515, -1.420356273651123, 0.32995402812957764, 0.20791487395763397, 0.2902017831802368, -0.872597336769104, -0.0734647884964943, 1.2763692140579224, 1.1456458568572998, -0.6159758567810059, 1.5986419916152954, -0.42394086718559265, -0.900426983833313, 1.1188578605651855, 0.3221486508846283, 2.287144184112549, -0.23529130220413208, 0.130361869931221, 0.7315894365310669, 0.26089879870414734, -0.404752254486084, -0.00014914169150870293, -0.024180471897125244, -0.022056881338357925, -0.03220084309577942, -1.5400729179382324, 0.8898439407348633, -0.6216471195220947, -0.2475295066833496, -0.7918187975883484, -0.2088126242160797, 0.9753791689872742, 1.2647300958633423, 0.32301634550094604, 0.1623605191707611, -0.5211617946624756, -0.28005844354629517, 0.23410500586032867, -0.15567798912525177, -0.98775315284729, 0.06309320032596588, -1.0450475215911865, 2.0714614391326904, 0.3069778084754944, -0.5659050941467285, 0.4661906063556671, -0.7466537952423096, -0.2525039613246918, 1.5514439344406128, 1.2888000011444092, -1.18357253074646, -1.1166119575500488, -0.5127487778663635, 2.254312753677368, 0.06251634657382965, 0.2586163580417633, -0.5088662505149841, 1.3280693292617798, 0.9648568630218506, -1.7288734912872314, -0.8472025394439697, -1.8318668603897095, -1.161454200744629, -0.8155563473701477, -1.337268590927124, -0.6383289694786072, 0.6911443471908569, 1.1265125274658203, 0.11244478076696396, 0.727006196975708, -1.357144832611084, 1.473545789718628, 2.2940237522125244, -0.14565110206604004, 0.5715117454528809, 0.35586097836494446, -2.1712448596954346, 1.0181353092193604, 0.47748786211013794, -1.622186541557312, -0.42135509848594666, 1.1579185724258423, 0.5736400485038757, -1.3796759843826294, -0.5750095844268799, -0.6916842460632324, 0.1504015326499939, -0.709019124507904, 0.04529527574777603, -0.17717322707176208, -1.486929178237915, 0.40389761328697205, 0.25397565960884094, -0.6908780932426453, 0.7372959852218628, 2.288269281387329, -1.59195876121521, 0.21407660841941833, 0.8331859707832336, 0.027106601744890213, -0.5867118239402771, 2.19681715965271, -0.5090339183807373, -0.1368144452571869, -0.6610105633735657, 1.2120517492294312, -2.1501693725585938, -0.9033632874488831, -0.4639155864715576, 0.30967921018600464, 0.34024083614349365, 0.9874232411384583, -0.12924598157405853, -0.7421562075614929, -0.06890545785427094, -0.8775032162666321, -0.5418280363082886, -0.48497700691223145, 0.3012404143810272, 0.8289187550544739, -1.1080992221832275, -0.6012536883354187, -0.029587820172309875, 0.00908153411000967, -0.2596210837364197, 0.5727977752685547, -0.22252051532268524, -1.750677227973938, 1.3114477396011353, -1.2555367946624756, 1.6305958032608032, 0.25769829750061035, 0.3203674852848053, -1.4713044166564941, 0.2672349214553833, -0.20969049632549286, -0.8428940176963806, 0.5448451638221741, 0.7449153661727905, 1.1834092140197754, -1.2243996858596802, -0.78768390417099, 1.3828811645507812, 0.5226479172706604, 0.8078532814979553, -0.520504355430603, 0.3545316457748413, 0.8162643909454346, -1.210927128791809, -0.6695255041122437, 0.25606769323349, 1.377608299255371, -0.4113657474517822, 0.6757513880729675, -0.8163938522338867, -1.1636176109313965, 0.6391131281852722, -0.9952322244644165, 0.05815848335623741, -0.533484160900116, 0.5298131704330444, -1.2293165922164917, -1.3673467636108398, -1.383439302444458, -0.18161441385746002, 1.2800629138946533, 0.8276116847991943, 0.20504823327064514, -0.12600857019424438, 1.2111464738845825, -0.08900467306375504, 1.0381709337234497, 1.3474973440170288, 0.6778573393821716, 0.3986591398715973, 0.39027076959609985, -0.07501524686813354, 0.2824588716030121, -0.8621513247489929, -0.8271771669387817, 1.546308994293213, -0.11820954084396362, -1.6666117906570435, 1.3354051113128662, 1.335755705833435, 0.44275519251823425, -1.2155958414077759, 1.9094475507736206, 1.0666695833206177, 1.213539481163025, -0.24352845549583435, 0.41056567430496216, -0.27121031284332275, -0.23361434042453766, -1.1505528688430786, -0.7728486657142639, -0.5033354759216309, 0.034832097589969635, 0.9672147035598755, 0.5426768064498901, -1.7780215740203857, 1.0147168636322021, -1.4898219108581543, 0.4711143672466278, -0.20116931200027466, 0.05644823983311653, -0.4806569218635559, -1.0501283407211304, 1.587028980255127, -2.42057466506958, -0.8879384994506836, 1.2868201732635498, -0.02850174345076084, -0.0005164616159163415, -0.8607780933380127, -1.1333407163619995, 0.5662614107131958, -0.1874246746301651, 0.31392809748649597, 0.7365867495536804, -1.717223882675171, -1.0469468832015991, -0.6849700808525085, 0.19882072508335114, 0.6641921401023865, -1.0488983392715454, -0.24053502082824707, -0.13416704535484314, -0.1357347071170807, 1.5181041955947876, -0.7604106664657593, 0.13331857323646545, -2.3214564323425293, -0.14034903049468994, 0.08891113847494125, 0.6703552603721619, -0.5351865291595459, -0.5813819766044617, -1.2245761156082153, -0.5248753428459167, 2.3303475379943848, -1.7258176803588867, -1.717647671699524, 0.06859401613473892, -0.96378493309021, 1.2066230773925781, -1.521357536315918, 1.6101632118225098, -0.4663284718990326, -1.6712396144866943, -2.2306129932403564, -0.9223074913024902, -0.670036256313324, 0.33444952964782715, -1.3074172735214233, 3.007284164428711, 0.8112484216690063, -0.20738321542739868, -0.5748499035835266, 2.662313222885132, -0.7807251214981079, 0.7354955673217773, 1.4572885036468506, -0.9418749213218689, 0.5675895810127258, -0.5077329874038696, 1.3750652074813843, 1.4414803981781006, 0.5370259284973145, 0.1561291515827179, -1.2203924655914307, 2.4406538009643555, 1.5092365741729736, 0.7196372151374817, 1.4974257946014404, 0.02074328623712063, -1.1149388551712036, 0.9441384673118591, -0.9607442021369934, -0.5270904302597046, 0.755734384059906, -1.5669032335281372, 0.03236565366387367, -0.29129818081855774, -1.1582468748092651, -1.2087905406951904, -0.21525515615940094, -0.29220297932624817, 0.16288518905639648, -1.4940520524978638, 0.08669523894786835, -1.261232852935791, -0.808051347732544, 0.970258355140686, -0.6791268587112427, -0.49367374181747437, -0.8134999871253967, -0.008317743428051472, 0.8369407653808594, -0.34862837195396423, 0.6661301255226135, -1.304329752922058, 0.4624706208705902, -0.10927121341228485, 0.07361458241939545, -0.9249707460403442, -0.6163270473480225, 0.3085590600967407, -1.1815680265426636, 0.7706934213638306, -0.4389411211013794, 0.45316174626350403, -1.1293416023254395, -1.6239328384399414, 0.6404016613960266, 0.6862847805023193, -0.0427534319460392, 0.6546756625175476, -0.012721925973892212, 0.6603957414627075, 0.146528959274292, -0.967114269733429, 1.1055102348327637, -0.9620632529258728, 2.2594985961914062, -0.14664633572101593, -0.9268412590026855, -0.2125709056854248, 0.0667780339717865, 0.14105013012886047, 0.5644569993019104, -0.5902034044265747, -2.0508666038513184, -0.8443393111228943, 0.44546768069267273, 1.129862666130066, -1.977871298789978, 0.1170620545744896, 1.1321496963500977, -1.3894414901733398, 1.7259252071380615, -0.7526829838752747, -0.09808292239904404, -0.4762236773967743, 1.805836796760559, 0.9198819994926453, 0.15647941827774048, -0.3363763988018036, -1.6679317951202393, -0.2736459970474243, -0.5885868072509766, -0.10653622448444366, -1.9005191326141357, 1.2882838249206543, 0.9321426749229431, 1.62429940700531, -0.5695050954818726, -0.017534788697957993, -1.2107858657836914, 0.9083598256111145, -0.2029878944158554, 1.7004809379577637, 0.7572497129440308, 0.7947508096694946, 0.5733243227005005, -1.4585809707641602, 0.7537329792976379, -1.591161847114563, 0.3259855806827545, 0.6174169778823853, -1.1374808549880981, -0.6102640628814697, -0.1519741266965866], "dtype": "float32"}], "e": {"__nd__": [-1.768783450126648, -1.801685094833374, -0.7965892553329468, 2.3428165912628174, 1.3230936527252197, 0.9799439907073975, -0.8213992118835449, 2.590074062347412, -1.1502565145492554, -2.1912484169006348, 1.5732603073120117, -2.454124927520752, -0.46228790283203125, -1.2908613681793213, 0.8949850797653198, -0.3752785921096802, -0.4587553143501282, 2.7854161262512207, -1.5598256587982178, 1.070492148399353, -1.0733413696289062, 1.203539490699768, 1.8743987083435059, 1.1255820989608765, -0.798451840877533, -0.7128324508666992, -0.32421860098838806, 1.0556029081344604, 1.3611173629760742, 1.9701687097549438, -1.0307550430297852, 0.15710675716400146, -0.6075582504272461, -0.5603172779083252, 1.167771577835083, -1.4513736963272095, 0.17428839206695557, -1.6588518619537354, 0.501503586769104, -1.5365121364593506, 0.05658745765686035, 0.3230155110359192, -0.7431778907775879, -2.3050897121429443, 1.9931292533874512, 0.26034361124038696, 1.4411957263946533, -2.478121757507324, 1.577686071395874, -0.16143798828125, 0.405648797750473, 1.4327683448791504, 0.3584684133529663, -1.6014995574951172, -1.1572984457015991, -0.5878661870956421, 2.7756152153015137, 0.6376754641532898, -0.32097554206848145, -0.5219802856445312, -0.03266000747680664, -1.3593008518218994, 0.9553182125091553, 0.8282431364059448, -0.3380739688873291, -2.3968517780303955, 0.06263640522956848, 1.5426613092422485, 3.076310396194458, 0.602928638458252, 2.6920223236083984, 0.7780667543411255, -0.2342575043439865, -2.8900957107543945, 1.4313428401947021, 1.107064127922058, 0.04335525631904602, 0.18213629722595215, 1.3234803676605225, -3.537200927734375, 0.5727573037147522, -2.1143548488616943, 1.549409031867981, -1.5588887929916382, 0.31882089376449585, -0.7106454372406006, 0.3679887652397156, -0.7907721996307373, 0.6289240121841431, 0.6277761459350586, -4.149806499481201, 0.1652752161026001, 0.06425511837005615, -3.0831432342529297, -1.5198750495910645, -0.236772358417511, -2.2230238914489746, 0.8030663132667542, -1.221473217010498, 0.7065025568008423, -2.756833791732788, 0.681625247001648, -0.5774294137954712, 0.8185285925865173, -0.19692835211753845, -0.11288285255432129, -0.019297361373901367, -0.17339646816253662, -1.2554019689559937, -1.886536717414856, -0.6944501399993896, 1.6447981595993042, -0.24113352596759796, -0.09254705905914307, -1.2551201581954956, -0.6108748316764832, 1.6976120471954346, -2.2708961963653564, 2.7091574668884277, -0.3928053379058838, 4.2414116859436035, 0.6213778257369995, -1.1347424983978271, 1.6259512901306152, -0.3836747407913208, 0.7932460308074951, -0.66098952293396, -1.2744572162628174, 0.5971859693527222, -0.26646366715431213, 0.37239277362823486, 0.00576627254486084, -0.3841817378997803, -0.6548437476158142, 0.23895639181137085, -0.21778756380081177, 0.5557899475097656, -1.5090281963348389, -0.8028921484947205, -1.5444647073745728, 0.6178802847862244, -0.2957414984703064, 1.5039305686950684, 0.41992002725601196, 0.5070067644119263, -3.161314010620117, -1.0653070211410522, -3.961264133453369, 1.5443896055221558, 0.209509015083313, 0.9857450723648071, -1.5075383186340332, -0.11121153831481934, -2.916327953338623, -0.504066526889801, -1.4809163808822632, -2.027015209197998, -0.13931381702423096, -0.05714726448059082, 0.1289641112089157, -0.957612156867981, -1.004666805267334, 0.5144529938697815, 0.4210367798805237, -1.522660255432129, 0.24521677196025848, -0.12166722118854523, 1.090078353881836, -1.9935530424118042, 2.212900400161743, -2.3710274696350098, -0.1326572299003601, -0.48495030403137207, -1.9202327728271484, -1.265130639076233, 0.7548718452453613, -1.5183088779449463, -0.4440654516220093, 0.8315013647079468, -2.292649745941162, -0.32773643732070923, 0.6677996516227722, -0.7120720744132996, 1.6730196475982666, 0.6108342409133911, 0.7620168924331665, 0.41600191593170166, 0.9462329149246216, -0.08607275038957596, 1.1159381866455078, -0.8378909826278687, -0.0075113773345947266, 0.9880795478820801, -0.6743078827857971, -1.3990968465805054, -1.1147197484970093, -1.463904619216919, -0.7851670384407043, 0.40907666087150574, -0.3498992919921875, -2.0078182220458984, -0.4596688747406006, -0.8656185865402222, -0.6304032802581787, 1.6653660535812378, -0.1725810319185257, -1.188018560409546, 2.2552595138549805, -0.7239519357681274, -2.0023722648620605, -2.698233127593994, -1.7951213121414185, 0.27414458990097046, -1.0970999002456665, -2.818490982055664, 0.8810393810272217, -1.2758833169937134, -0.9724681377410889, 1.296799659729004, 0.360032320022583, -1.8039908409118652, -0.04288512468338013, 0.2536850571632385, 0.261336088180542, 1.6320233345031738, -1.0022075176239014, -0.9553585648536682, -3.3426289558410645, 1.920164942741394, -1.9508165121078491, 0.9236729145050049, -1.0591915845870972, 0.30487263202667236, 0.5363302826881409, -1.0010383129119873, -0.15199780464172363, -1.9192512035369873, 1.8568627834320068, 0.24914449453353882, -2.0857744216918945, -0.2850395441055298, 0.15906403958797455, 0.9009437561035156, -0.12295413017272949, 0.5661433339118958, -1.9079720973968506, 0.1756260097026825, 0.029358744621276855, 0.19653046131134033, -1.3423107862472534, -1.0546581745147705, 0.5520884990692139, 1.07224702835083, -0.5089985132217407, 0.7320998907089233, -1.6089200973510742, 0.8465157747268677, 0.1894248127937317, -0.6207215785980225, 1.3915934562683105, -0.1991811990737915, 1.2867618799209595, -0.1146230697631836, 2.1481075286865234, 0.4048263430595398, -2.2299275398254395, 0.2003019154071808, 0.7827253937721252, 0.4931810200214386, -0.15892183780670166, 0.3733120858669281, 0.2956431806087494, 1.7184159755706787, 0.780684769153595, 1.858202576637268, -1.9194581508636475, -0.883564829826355, -2.0757369995117188, 0.8355478048324585, -0.598301887512207, 0.1832883358001709, -1.6005027294158936, -0.34722644090652466, 0.733497142791748, -0.2615275979042053, 0.5350930094718933, -1.237410306930542, 2.3987154960632324, -1.383685827255249, 0.30066877603530884, -3.289523124694824, -0.177129864692688, -1.4237186908721924, -1.3515958786010742, 0.43242284655570984, -1.1844979524612427, 2.7641375064849854, 0.5380681157112122, -2.6928839683532715, -0.36789917945861816, -2.7358555793762207, 2.8560986518859863, 1.3392966985702515, 1.417151689529419, -0.7393039464950562, -1.6937141418457031, -1.4982391595840454, -0.2251218557357788, -0.20309287309646606, -1.3802220821380615, -0.6774322390556335, 0.2234344780445099, 1.0717111825942993, 3.0517125129699707, 0.7428979277610779, 0.3921026587486267, -1.1967530250549316, -1.199646234512329, -0.4773324131965637, 1.0189074277877808, 2.3281052112579346, 0.8300975561141968, -0.1796790361404419, -1.2369592189788818, -2.3674001693725586, -0.21023590862751007, -1.3947304487228394, 2.441826581954956, 0.7091145515441895, -0.5330543518066406, 0.6248800754547119, 1.258303165435791, 1.0154403448104858, -0.4761385917663574, 1.3166522979736328, -0.8161937594413757, 1.8279566764831543, 1.748733401298523, 0.3436192274093628, -2.5369534492492676, 2.1147170066833496, -0.49868255853652954, 1.1485234498977661, 0.6403160691261292, 1.953635573387146, 1.6935291290283203, -1.18403160572052, 0.916917622089386, -0.2830323576927185, -1.6509212255477905, -1.159104347229004, 1.123478889465332, -1.8358453512191772, -0.0033310651779174805, 0.3067106604576111, -0.35228580236434937, 0.727191686630249, -0.857285737991333, 0.9886611700057983, -0.003528360277414322, 0.3944057524204254, 1.4886889457702637, 2.0007569789886475, -0.4508238434791565, -1.7732738256454468, -1.2084439992904663, 0.4322178363800049, 0.3886783719062805, -0.7368720769882202, -1.3062916994094849, -2.294490098953247, 1.1900622844696045, -0.41667479276657104, -0.3299865126609802, 1.6439716815948486, -0.6255030632019043, -0.040145665407180786, -1.3469876050949097, -2.147976875305176, 0.4857058823108673, 0.6618334650993347, -0.6435384154319763, -2.2464301586151123, 0.48713552951812744, -3.4885475635528564, -2.1492919921875, -1.4545462131500244, 0.24899959564208984, -0.09795570373535156, 0.10320361703634262, -0.6695785522460938, 0.7306523323059082, 2.108762264251709, 1.6130342483520508, -0.01762029528617859, 1.2771947383880615, -2.741119384765625, 0.411421000957489, -0.1841590255498886, -2.2373788356781006, -0.42441585659980774, 1.9948699474334717, -0.8811906576156616, -1.7515708208084106, 3.385319709777832, -0.01156628131866455, -0.6356150507926941, 1.978545904159546, -3.170442581176758, 0.07808683812618256, 1.0484120845794678, 3.2095651626586914, 0.19567495584487915, -1.2019712924957275, -1.0379388332366943, -0.5494327545166016, 0.939568042755127, 0.1407281756401062, 0.9294663667678833, 2.593852996826172, -0.20893773436546326, -0.1120944619178772, 1.8121997117996216, 0.5935875177383423, -1.1541662216186523, -1.6536953449249268, -0.9284166097640991, -2.266866445541382, 1.3238309621810913, 0.3447016477584839, -2.203395366668701, -1.3915271759033203, -0.12820088863372803, -1.4823870658874512, -1.0217227935791016, 0.5668194890022278, 1.37346613407135, -0.8715825080871582, -0.4071058928966522, -1.6009981632232666, -0.7702961564064026, 2.532947540283203, -0.9390333890914917, -1.8385463953018188, -0.09766632318496704, -1.2346242666244507, -2.7113330364227295, -3.2941389083862305, 3.0419461727142334, -1.7248954772949219, -1.6739404201507568, 1.712335467338562, 1.655631422996521, -0.15457457304000854, -1.7794537544250488, -0.5176888704299927, -3.1764307022094727, -0.31918445229530334, -1.7262732982635498, -0.10885822772979736, -2.00537371635437, -0.3379611670970917, 0.4691660404205322, 3.28232479095459, -0.7739471197128296, 1.2034225463867188, 1.0269393920898438, -1.994720458984375, -0.5860929489135742, 1.7944092750549316, -2.1127102375030518, -0.5726022720336914, -0.7672277688980103, -0.46782657504081726, 0.21341627836227417, 1.9666328430175781, -1.0727262496948242, 0.5427133440971375, -0.4831579923629761, -0.8034166097640991, 2.437886953353882, 0.018736593425273895, 2.3777449131011963, -1.2976676225662231, -2.2727363109588623, -0.6197584271430969, 0.8552799224853516, -0.3125889301300049, 0.1325809359550476, 0.5615105628967285, 2.7236886024475098, -0.06884351372718811, 0.04804021865129471, 1.035484790802002, -0.5365555286407471, -1.6780409812927246, 0.2296888530254364, 2.402801036834717, 1.3539174795150757, 2.242220401763916, 0.23295144736766815, -0.9984524846076965, -2.1778993606567383, 0.5096790194511414, 1.1176856756210327, -1.895326018333435, 0.6368341445922852, -2.44714093208313, 1.6048283576965332, -0.8379198312759399, -1.552786111831665, -0.058042287826538086, -0.21662777662277222, 1.997229814529419, 1.9692022800445557, -0.5618799924850464, 3.3485515117645264, -0.4820559024810791, 0.9219419956207275, 1.775364637374878, -2.5190823078155518, 2.846073627471924, 1.0189708471298218, -0.7718366980552673, 1.2531137466430664, 1.1645256280899048, -0.200261190533638, 1.1626214981079102, -1.2951346635818481, 1.9861552715301514, 0.4743807911872864, -0.5099895596504211, 1.0666779279708862, -3.2375807762145996, -4.213663101196289, 0.44105151295661926, -1.5260508060455322, -1.5871751308441162, 0.35217493772506714, 0.9473400115966797, 0.6363251209259033, 2.613168954849243, -0.8535317182540894, -0.8913291096687317, 0.7190366983413696, 1.0958795547485352, -1.6408016681671143, -0.2507386803627014, 3.5839271545410156, 1.4201374053955078, 1.9302878379821777, 2.3553566932678223, -1.9469101428985596, -1.9115862846374512, 2.3461265563964844, -0.06545156240463257, 0.9108583927154541, -1.342834711074829, 1.9084116220474243, 0.7219783067703247, 2.2313311100006104, -3.2754077911376953, -0.8059674501419067, 2.6309213638305664, -1.2235628366470337, -0.15907898545265198, -0.5419510006904602, -0.1982353925704956, -2.953503370285034, -0.5640133619308472, -0.5606386661529541, 1.5850709676742554, 1.7582603693008423, -0.4790249466896057, 0.04665932059288025, -1.106003999710083, -0.4037524461746216, 0.9473417401313782, -0.5972569584846497, 0.32176801562309265, -0.3229246139526367, 0.7011579275131226, 0.5409431457519531, -0.09574451297521591, -0.40175509452819824, -0.030426621437072754, -2.0449047088623047, 2.4543745517730713, -1.3961352109909058, 0.34348827600479126, -1.674426555633545, 0.3270327150821686, 1.3177862167358398, -1.781663179397583, 0.6987356543540955, 1.7057591676712036, -0.5011653900146484, -0.42270493507385254, -1.2179343700408936, 1.9778774976730347, -1.0683258771896362, 2.2894606590270996, -0.5084095597267151, 0.20036429166793823, -1.3804402351379395, 2.512575387954712, 3.920315742492676, 0.9456299543380737, -0.07876074314117432, 1.0777028799057007, 1.6887420415878296, -0.7294667959213257, 0.5542212128639221, -2.605877637863159, 2.3110780715942383, -0.3063117265701294, -0.6782840490341187, 0.06046232581138611, -2.3416032791137695, -1.748155951499939, 1.0623061656951904, 0.09459182620048523, 0.6873927116394043, -0.44942447543144226, -2.484269618988037, 0.945163905620575, -0.5263979434967041, -1.4042061567306519, -2.011270046234131, -1.4804284572601318, 0.8570101261138916, 0.18117469549179077, -0.7820529341697693, 0.5253168940544128, -0.24316740036010742, -1.4649617671966553, 0.5737455487251282, -2.6262805461883545, -1.2459770441055298, 0.040406763553619385, 1.6347719430923462, -1.585374355316162, -0.4841434955596924, -1.5120006799697876, -0.3111628293991089, -0.3765147626399994, -0.8706657886505127, -1.5020320415496826, 1.10148286819458, -0.6758562326431274, 0.9006310701370239, 0.8618117570877075, -1.0890790224075317, 0.48031145334243774, -4.422536373138428, 1.7943708896636963, -1.799018383026123, -2.619537830352783, -3.4908270835876465, 0.6403193473815918, 1.634196400642395, -1.1454278230667114, -0.5686115026473999, -0.23084090650081635, 0.7648255825042725, -0.5001478791236877, 1.821707010269165, 0.6868452429771423, 0.19622910022735596, 0.4787330627441406, -1.2881428003311157, 0.008120745420455933, -0.9188107848167419, -1.0781384706497192, 0.2649657130241394, 1.8007460832595825, 0.3592537045478821, 0.8169993758201599, -1.737914800643921, -1.3109792470932007, 0.07275138795375824, -1.6091686487197876, -0.3800586462020874, -0.5123751759529114, -0.031246185302734375, 2.4019243717193604, 2.1357874870300293, -0.3954213857650757, 1.8359980583190918, -0.5067306756973267, 0.4676942825317383, -0.5486776828765869, 1.4192814826965332, -0.47769051790237427, -0.6086347699165344, 0.7708383202552795, -2.081087827682495, -1.874011754989624, -0.04677385836839676, -1.3126397132873535, 0.6267382502555847, 2.9857616424560547, 0.5447912812232971, 1.1096289157867432, 0.28442084789276123, -0.3423309922218323, 0.5762535333633423, -0.05436122417449951, 0.5901948809623718, -0.9420895576477051, 0.5731397867202759, 1.3057208061218262, -2.6926331520080566, -2.2924752235412598, -0.9222285151481628, 0.3949018716812134, -0.09352541714906693, -2.9830989837646484, 0.1692521870136261, 0.6239827871322632, -2.5280017852783203, 1.0712072849273682, -0.7906385064125061, 1.0373140573501587, -1.613700270652771, 1.5661007165908813, -1.6149492263793945, 0.15852110087871552, -0.831608235836029, -0.23752129077911377, -0.35129421949386597, -0.09147286415100098, 1.4451113939285278, -2.047287940979004, -0.8778131604194641, -2.0695252418518066, -2.161886215209961, 0.05289772152900696, 1.3536298274993896, 0.07977151870727539, 1.637279748916626, -2.0822625160217285, 0.07325553894042969, 0.1687416434288025, -1.1813602447509766, -0.22775673866271973, -1.8251097202301025, -1.577721357345581, -1.0350680351257324, 0.5400027632713318, 0.6383880376815796, -2.289599895477295, -1.1252347230911255, -0.7585231065750122, 0.7806534767150879, 0.7838983535766602, -2.6526083946228027, -0.011532217264175415, 0.34813159704208374, 0.9850460290908813, -1.4299938678741455, -0.13600793480873108, 0.7326956987380981, 0.49311739206314087, 0.45320630073547363, 1.0998343229293823, 1.192861557006836, 2.5610852241516113, -2.1196115016937256, -1.3126411437988281, -1.7731647491455078, -1.5151138305664062, -2.0315935611724854, -1.5314613580703735, 1.220654010772705, -0.04785078763961792, -1.0449162721633911, -0.1461268961429596, -0.35780155658721924, 1.7172856330871582, -1.7150838375091553, -1.0544172525405884, 0.9430862069129944, 2.0687386989593506, 1.690906286239624, -2.13802170753479, -1.7065820693969727, 1.0693647861480713, -0.7235287427902222, -2.5654828548431396, 0.9298195838928223, 0.0033221542835235596, 0.4497677981853485, -0.09104570746421814, 1.7093863487243652, -1.8944816589355469, -0.7082318067550659, 0.10035312175750732, 0.5393252372741699, 0.9851522445678711, -0.5271362066268921, 2.6941585540771484, 0.022858023643493652, -1.2243441343307495, -0.7900237441062927, 1.0706735849380493, -3.8726882934570312, -1.1745696067810059, 2.2175657749176025, 0.20016802847385406, -2.236412286758423, 0.4428935647010803, -1.6360841989517212, 1.3808279037475586, -3.590216875076294, 1.010207176208496, -2.3431801795959473, -1.4926855564117432, 0.34878385066986084, -0.33924782276153564, -0.6712672710418701, 0.8779534101486206, 0.7438159584999084, -0.6850696802139282, 1.3120187520980835, 0.7114686965942383, -2.6177115440368652, 2.861703395843506, -0.31237393617630005, -1.1911590099334717, 0.7185436487197876, 0.6222066283226013, -1.264038324356079, 1.5438737869262695, -1.1578105688095093, -1.6141364574432373, -0.4964444637298584, -0.5807743072509766, 0.6306622624397278, 0.0174483060836792, -0.6770036220550537, -0.10913556814193726, -0.6847342252731323, 0.7255147695541382, -1.4193922281265259, 0.03880813717842102, 1.656980276107788, -0.8196943998336792, 1.0197855234146118, -0.6846042275428772, 0.8229738473892212, 0.1699894815683365, -1.8504760265350342, -0.1663430631160736, 2.0181024074554443, 2.865156650543213, 0.278331995010376, 2.4702858924865723, -0.11898656189441681, -0.08285403251647949, 0.5786209106445312, -0.9824528694152832, 0.9899934530258179, -0.03913611173629761, -0.8163295984268188, -2.192636013031006, 0.6629665493965149, -0.10131490230560303, -0.5976957082748413, 0.21332582831382751, 1.6445744037628174, -0.512253999710083, -0.7530455589294434, 1.1231619119644165, -0.8096756935119629, 0.48162806034088135, 0.2788725793361664, -0.18628095090389252, -0.03616535663604736, 0.6475622057914734, -0.6096380949020386, -0.653631329536438, -3.234412431716919, 0.5538498759269714, 2.269594669342041, -0.5275918841362, -2.8935797214508057, -1.34022855758667, -0.8185575008392334, -0.04873394966125488, 1.2423300743103027, -0.08817693591117859, 1.2377076148986816, -0.5704643726348877, -0.5492510795593262, 0.5148380398750305, -0.34493646025657654, 2.8956384658813477, -0.3748648464679718, 0.3381333351135254, 1.377578854560852, -1.380509376525879, -0.9663605690002441, 0.7453844547271729, -3.518244743347168, -2.45601487159729, -0.6771247982978821, -0.5790508389472961, -1.2842624187469482, -2.266710042953491, -0.23067069053649902, 0.8247134685516357, -0.28444063663482666, -1.8730448484420776, 1.117903470993042, -1.774484395980835, 1.3378753662109375, 1.3611030578613281, 0.1688191443681717, -3.435570240020752, -0.5789817571640015, -0.1956152766942978, -0.2939208745956421, -1.7672955989837646, 0.8700255155563354, 0.9523659348487854, -1.7522664070129395, 1.965393304824829, -0.8702234625816345, -2.1188035011291504, 0.0816851258277893, 0.09674704074859619, -0.3799600601196289, 0.9064170122146606, -1.5038952827453613, -1.3946497440338135, 0.5668529272079468, 1.1522879600524902, -0.6656430959701538, 0.7162650227546692, 1.0983467102050781, -0.9948900938034058, -0.7910750508308411, 0.5238879323005676, -1.0952186584472656, 0.5630003213882446, -1.4611839056015015, 1.1106287240982056, 2.2648425102233887, -1.6607487201690674, -2.0465588569641113, 1.6035609245300293, -2.637057304382324, 2.464298725128174, -0.5010995864868164, 1.3746201992034912, -1.5502958297729492, -0.5432420372962952, -1.013136386871338, -3.859679698944092, -0.5206568241119385, -2.047133445739746, 1.5000433921813965, -0.8907508850097656, 0.6205362677574158, 2.247525691986084, -0.5363056063652039, 0.2663019299507141, -0.5332285165786743, -0.25581467151641846, 1.3163561820983887, 1.312532901763916, 1.1359139680862427, 0.831895112991333, -0.2223045825958252, 0.8695389032363892, -1.0207276344299316, 1.8054356575012207, 0.3396967053413391, 1.7207324504852295, 1.7160271406173706, 2.6785526275634766, -1.463529348373413, -1.1481212377548218, 0.6590161919593811, -2.516784429550171, 1.3564519882202148, -0.06212174892425537, -0.3690353035926819, -1.4865769147872925, -0.8211368918418884, 0.87694251537323, 1.410391926765442, 1.2130793333053589, 3.8585636615753174, -2.1982123851776123, 0.08928892016410828, -0.5039110779762268, -3.5784902572631836, 0.7714426517486572, -0.020565152168273926, 0.31174376606941223, 1.7336673736572266, 1.0742111206054688, -1.01039457321167, 1.9877276420593262, 0.7639784216880798, -0.41538065671920776, 0.44006675481796265, -0.6196396350860596, 0.3908882737159729, -0.6964266896247864, 0.9457644820213318, 1.7387471199035645, 0.8891014456748962, -0.44041216373443604, -0.3822721242904663, 0.3372493386268616, 0.014713704586029053, 1.4052836894989014, -0.9527997374534607, 4.845847129821777, -1.947095513343811, 1.3254845142364502, 1.1225175857543945, 0.8275849223136902, -1.3174724578857422, 0.6586307287216187, 1.2264683246612549, -2.1797518730163574, -1.620308756828308, 1.9801926612854004, 2.0002200603485107, -0.15319490432739258, -2.3214006423950195, -1.0141867399215698, 1.712942123413086, -0.8378003239631653, -3.06565523147583, -0.0826272964477539, 0.8209735155105591, 0.543779730796814, -0.950057864189148, 1.2141172885894775, -0.5870847702026367, -0.3249114751815796, 0.10709448158740997, 1.4683966636657715, 0.9152071475982666, 2.416454315185547, 1.774765133857727, -0.7186424732208252, -1.1994014978408813, 0.41476258635520935, 0.6530865430831909, -0.244968980550766, -2.389915943145752, -0.14671432971954346, -1.928328514099121, 1.7054985761642456, 0.40668678283691406, -0.6767228245735168, 0.8314231038093567, 0.0011864304542541504, 1.362330675125122, -3.2302165031433105, -0.8909035325050354, 1.697096347808838, 1.6777915954589844, -0.5970485210418701, 0.37670475244522095, 1.3899095058441162, -0.7773334383964539, -3.1376092433929443, -2.4537317752838135, 0.1186748743057251, -0.5810497999191284, 0.1532384157180786, 1.5708343982696533, 0.19617223739624023, -3.3555479049682617, -0.5917568206787109, 1.0191457271575928, 1.4522473812103271, 1.3034721612930298, -2.9715843200683594, 0.6042754650115967, 2.8858182430267334, -1.6327422857284546, -2.705003261566162, 2.1144747734069824, 0.061046600341796875, -0.6527401208877563, 0.6807355880737305, 2.042233467102051, -0.7051506638526917, -0.5636617541313171, -0.04076516628265381, -0.5497710704803467, -0.6705647110939026, -0.08450281620025635, -0.7793350219726562, -2.0642318725585938, 0.0070105791091918945, 0.6816744804382324, -0.06332129240036011, -0.758001446723938, -1.3294854164123535, -0.2523573637008667, -0.40424972772598267, 1.3080921173095703, 0.24770385026931763, 1.31839919090271, -1.2793947458267212, -0.6244306564331055, -0.9576110243797302, 1.729475498199463, -0.23476219177246094, -1.804693579673767, 0.35005486011505127, 0.883755087852478, -1.1534967422485352, -0.44830748438835144, 1.6281018257141113, 0.4351680874824524, 0.3444169759750366, -0.9864292740821838, -1.6996663808822632, 1.0457289218902588, -1.9547781944274902, 0.3934137225151062, -1.1440445184707642, 2.183581829071045, -1.516268014907837, 0.12928879261016846, -2.0306098461151123, -0.27778273820877075, -1.3063195943832397, 1.323628544807434, 0.8573799729347229, -1.4792805910110474, 1.6728177070617676, -1.7804105281829834, -0.970628559589386, -0.3962433934211731, 0.9609668254852295, 0.0671890377998352, 1.2293723821640015, 2.0342447757720947, -0.5732296705245972, 0.39492151141166687, -0.6379069089889526, 1.5321062803268433, -0.3536008596420288, -0.716422438621521, 0.39136239886283875, 0.9320838451385498, 1.2624537944793701, -1.5720983743667603, -2.5427043437957764, 0.06973767280578613, -1.2919247150421143, 0.7006183862686157, 0.8451451659202576, -1.1174246072769165, 0.02839505672454834, -0.05757951736450195, -2.914792776107788, -0.4218318462371826, -4.174206733703613, -0.3010217845439911, -0.517829179763794, -0.6230332851409912, 0.6841644048690796, 1.3041496276855469, -1.9139866828918457, -1.7063273191452026, 1.6751712560653687, 2.7207794189453125, 0.42105579376220703, -1.9265542030334473, -0.269614040851593, -0.9752764701843262, -1.0519630908966064, -1.4224258661270142, -1.2476489543914795, 1.4660379886627197, 2.822547435760498, 1.481586217880249, 1.4801198244094849, 0.4536471664905548, 2.508543014526367, 0.33622798323631287, 1.1630573272705078, -1.4004989862442017, 0.6559517979621887, 0.2843666076660156, 1.110222578048706, -0.6256567239761353, -2.238884449005127, -0.62370365858078, -1.253246784210205, -0.03044375777244568, 3.9532594680786133, -1.5184433460235596, 0.06073719263076782, 0.8058255910873413, 0.6704539060592651, -2.473653554916382, -1.371835708618164, 1.3644853830337524, -0.18355894088745117, 0.4123021960258484, 0.853260338306427, 0.5757180452346802, -0.7726141214370728, -1.0676851272583008, 0.9460950493812561, 0.7431157827377319, -0.3418000638484955, 0.5380561351776123, -0.020384609699249268, -1.4551975727081299, 2.5919253826141357, -0.579522430896759, -1.757049798965454, -1.705249547958374, -0.10666871070861816, -2.5758938789367676, 1.2328448295593262, 0.4502049386501312, 0.08650362491607666, -0.3338624835014343, 0.8830398321151733, -0.1576281189918518, -0.722350001335144, -1.4116244316101074, -0.6038995981216431, -1.8440065383911133, -0.6421310901641846, 2.0718586444854736, -1.4085160493850708, -0.16660445928573608, 0.29219934344291687, 0.9180684685707092, 0.17475652694702148, -1.1054648160934448, 0.7294946312904358, 0.3233451247215271, 0.762624204158783, -0.9424304962158203, -2.10471773147583, -0.39883339405059814, -0.9590389132499695, 1.680160403251648, 0.06758642196655273, 1.3425076007843018, 0.1956552267074585, -1.860374927520752, -0.09242583811283112, 1.8507797718048096, 0.5525344014167786, -0.5301509499549866, 0.9937129616737366, -0.4322112798690796, -0.6636124849319458, -2.565061569213867, -0.5090992450714111, 1.3219408988952637, -1.041552186012268, -0.4902009069919586, -0.17413544654846191, 1.5984090566635132, -1.5193290710449219, -0.034984588623046875, 2.7029876708984375, 1.6950440406799316, 0.34010589122772217, -0.6650682091712952, -1.0220732688903809, -1.1597915887832642, 1.8552711009979248, 2.6615633964538574, -0.1603473722934723, 0.5599397420883179, -0.1741447150707245, 1.4109880924224854, 1.5166972875595093, 2.0490872859954834, 0.877326250076294, 2.0418341159820557, 0.6517075896263123, 1.1572473049163818, -0.19268196821212769, -0.12909331917762756, 0.09110577404499054, -2.399329662322998, -0.9228128790855408, -1.473188877105713, -2.1155829429626465, -0.4431876838207245, -0.25481849908828735, 0.5953936576843262, 1.6364450454711914, -0.08219268918037415, 0.7126231789588928, -0.9851841926574707, -1.119447946548462, 0.8256784081459045, 0.9429397583007812, -1.6805744171142578, -1.9888064861297607, 0.6628166437149048, 1.882201910018921, -0.6464347839355469, -2.2267956733703613, 1.4922336339950562, -0.5668898820877075, -1.391038179397583, 0.5035123825073242, -0.2794092893600464, 3.0085384845733643, 0.39978450536727905, -0.14378458261489868, 0.9784787893295288, 0.2913811206817627, -1.3460156917572021, 1.3343204259872437, -1.3414555788040161, 1.3609529733657837, 1.2714749574661255, 1.4897043704986572, 1.0398871898651123, 1.7841362953186035, 1.0955297946929932, -0.6871500015258789, 2.373544692993164, 0.16058707237243652, 0.4029442071914673, -1.7064323425292969, 0.025155020877718925, 0.3039628863334656, -0.5117483139038086, -0.9309430122375488, 0.2708011269569397, 0.1606215238571167, -1.5498030185699463, 2.4819326400756836, 1.364337682723999, 1.5048472881317139, -2.1172664165496826, -1.110353708267212, 0.43755483627319336, 0.8198161721229553, -0.6027510166168213, -0.3746474087238312, -2.660998821258545, -0.43525755405426025, -1.0704519748687744, 3.281783103942871, -0.9274924993515015, -2.530877113342285, -0.3886462152004242, 0.07171371579170227, 1.122239351272583, 1.746673822402954, 0.19808554649353027, -2.7652111053466797, 1.1577787399291992, 1.6931400299072266, -0.8623097538948059, 1.1424293518066406, 0.15856623649597168, 0.8384949564933777, -2.254343032836914, -0.18604609370231628, 2.2672669887542725, 2.8707377910614014, 2.3405680656433105, 1.3687877655029297, -0.21942150592803955, 1.750990390777588, -4.323803424835205, -0.25784826278686523, 0.3723863363265991, 2.6421844959259033, -1.2946134805679321, -0.048735082149505615, -1.3066030740737915, -0.584771454334259, -1.1276183128356934, -1.4891765117645264, 0.8728634119033813, -0.7357597351074219, -1.2601467370986938, 0.6223686933517456, -0.42979896068573, 1.3135831356048584, -2.635800838470459, -2.602969169616699, -0.6759933829307556, -2.049816131591797, 0.057912953197956085, -1.4972940683364868, -0.8363122344017029, 0.4179145097732544, -0.08779507875442505, -0.7112690210342407, 1.0954859256744385, 0.8926495909690857, 1.5364166498184204, 2.039841413497925, 2.171971559524536, 0.5786120295524597, 0.32440951466560364, -1.4419001340866089, 1.1267988681793213, 1.1229138374328613, 2.0264883041381836, 0.32068899273872375, -0.860497236251831, 1.7363165616989136, 0.334855318069458, 2.4341907501220703, -0.7450969219207764, -0.04646706581115723, 1.0451663732528687, -0.4905009865760803, 1.6088354587554932, 0.07910159230232239, -2.1514084339141846, 0.40007293224334717, 2.071629047393799, 1.0197885036468506, 0.8328237533569336, 0.007703065872192383, -0.8327364921569824, 0.7745396494865417, -0.7092244625091553, 2.3210902214050293, 2.1149582862854004, -1.1447856426239014, 0.13337701559066772, 0.864701509475708, -1.0842351913452148, 0.6439681053161621, -0.2776179015636444, 0.21270111203193665, -0.2766086459159851, 0.17226195335388184, 1.6133133172988892, -2.2804269790649414, 0.4990435838699341, 1.2660084962844849, -0.0867699384689331, -0.6454811096191406, -0.03930923342704773, -1.1548402309417725, 0.10974997282028198, 0.892379641532898, 0.5714077353477478, 0.5615661144256592, -0.9140152931213379, 2.027543544769287, -0.43045774102211, -1.555237889289856, -0.1031031608581543, -0.07211697101593018, 0.028347671031951904, -1.3985226154327393, 0.8769686222076416, -1.5517925024032593, -1.4373005628585815, -1.6286826133728027, -1.0419583320617676, 3.2345728874206543, 0.156279057264328, -0.0647881031036377, 1.9646484851837158, 1.3558523654937744, -2.212510585784912, -0.3884291648864746, -1.2578182220458984, 2.131748676300049, -1.5394258499145508, 1.4987058639526367, -2.603879928588867, 0.6632236838340759, -2.3554153442382812, -1.2197091579437256, -1.531280755996704, 0.26478809118270874, 3.0808866024017334, -0.8404762148857117, 1.3331294059753418, -0.8301403522491455, -0.09586107730865479, 0.1050952672958374, 4.153100967407227, -0.08259919285774231], "dtype": "float32"}}, {"a": [{"__nd__": [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0, 48.0, 49.0, 50.0, 51.0, 52.0, 53.0, 54.0, 55.0, 56.0, 57.0, 58.0, 59.0, 60.0, 61.0, 62.0, 63.0, 64.0, 65.0, 66.0, 67.0, 68.0, 69.0, 70.0, 71.0, 72.0, 73.0, 74.0, 75.0, 76.0, 77.0, 78.0, 79.0, 80.0, 81.0, 82.0, 83.0, 84.0, 85.0, 86.0, 87.0, 88.0, 89.0, 90.0, 91.0, 92.0, 93.0, 94.0, 95.0, 96.0, 97.0, 98.0, 99.0, 100.0, 101.0, 102.0, 103.0, 104.0, 105.0, 106.0, 107.0, 108.0, 109.0, 110.0, 111.0, 112.0, 113.0, 114.0, 115.0, 116.0, 117.0, 118.0, 119.0, 120.0, 121.0, 122.0, 123.0, 124.0, 125.0, 126.0, 127.0, 128.0, 129.0, 130.0, 131.0, 132.0, 133.0, 134.0, 135.0, 136.0, 137.0, 138.0, 139.0, 140.0, 141.0, 142.0, 143.0, 144.0, 145.0, 146.0, 147.0, 148.0, 149.0, 150.0, 151.0, 152.0, 153.0, 154.0, 155.0, 156.0, 157.0, 158.0, 159.0, 160.0, 161.0, 162.0, 163.0, 164.0, 165.0, 166.0, 167.0, 168.0, 169.0, 170.0, 171.0, 172.0, 173.0, 174.0, 175.0, 176.0, 177.0, 178.0, 179.0, 180.0, 181.0, 182.0, 183.0, 184.0, 185.0, 186.0, 187.0, 188.0, 189.0, 190.0, 191.0, 192.0, 193.0, 194.0, 195.0, 196.0, 197.0, 198.0, 199.0, 200.0, 201.0, 202.0, 203.0, 204.0, 205.0, 206.0, 207.0, 208.0, 209.0, 210.0, 211.0, 212.0, 213.0, 214.0, 215.0, 216.0, 217.0, 218.0, 219.0, 220.0, 221.0, 222.0, 223.0, 224.0, 225.0, 226.0, 227.0, 228.0, 229.0, 230.0, 231.0, 232.0, 233.0, 234.0, 235.0, 236.0, 237.0, 238.0, 239.0, 240.0, 241.0, 242.0, 243.0, 244.0, 245.0, 246.0, 247.0, 248.0, 249.0, 250.0, 251.0, 252.0, 253.0, 254.0, 255.0, 256.0, 257.0, 258.0, 259.0, 260.0, 261.0, 262.0, 263.0, 264.0, 265.0, 266.0, 267.0, 268.0, 269.0, 270.0, 271.0, 272.0, 273.0, 274.0, 275.0, 276.0, 277.0, 278.0, 279.0, 280.0, 281.0, 282.0, 283.0, 284.0, 285.0, 286.0, 287.0, 288.0, 289.0, 290.0, 291.0, 292.0, 293.0, 294.0, 295.0, 296.0, 297.0, 298.0, 299.0, 300.0, 301.0, 302.0, 303.0, 304.0, 305.0, 306.0, 307.0, 308.0, 309.0, 310.0, 311.0, 312.0, 313.0, 314.0, 315.0, 316.0, 317.0, 318.0, 319.0, 320.0, 321.0, 322.0, 323.0, 324.0, 325.0, 326.0, 327.0, 328.0, 329.0, 330.0, 331.0, 332.0, 333.0, 334.0, 335.0, 336.0, 337.0, 338.0, 339.0, 340.0, 341.0, 342.0, 343.0, 344.0, 345.0, 346.0, 347.0, 348.0, 349.0, 350.0, 351.0, 352.0, 353.0, 354.0, 355.0, 356.0, 357.0, 358.0, 359.0, 360.0, 361.0, 362.0, 363.0, 364.0, 365.0, 366.0, 367.0, 368.0, 369.0, 370.0, 371.0, 372.0, 373.0, 374.0, 375.0, 376.0, 377.0, 378.0, 379.0, 380.0, 381.0, 382.0, 383.0, 384.0, 385.0, 386.0, 387.0, 388.0, 389.0, 390.0, 391.0, 392.0, 393.0, 394.0, 395.0, 396.0, 397.0, 398.0, 399.0, 400.0, 401.0, 402.0, 403.0, 404.0, 405.0, 406.0, 407.0, 408.0, 409.0, 410.0, 411.0, 412.0, 413.0, 414.0, 415.0, 416.0, 417.0, 418.0, 419.0, 420.0, 421.0, 422.0, 423.0, 424.0, 425.0, 426.0, 427.0, 428.0, 429.0, 430.0, 431.0, 432.0, 433.0, 434.0, 435.0, 436.0, 437.0, 438.0, 439.0, 440.0, 441.0, 442.0, 443.0, 444.0, 445.0, 446.0, 447.0, 448.0, 449.0, 450.0, 451.0, 452.0, 453.0, 454.0, 455.0, 456.0, 457.0, 458.0, 459.0, 460.0, 461.0, 462.0, 463.0, 464.0, 465.0, 466.0, 467.0, 468.0, 469.0, 470.0, 471.0, 472.0, 473.0, 474.0, 475.0, 476.0, 477.0, 478.0, 479.0, 480.0, 481.0, 482.0, 483.0, 484.0, 485.0, 486.0, 487.0, 488.0, 489.0, 490.0, 491.0, 492.0, 493.0, 494.0, 495.0, 496.0, 497.0, 498.0, 499.0, 500.0, 501.0, 502.0, 503.0, 504.0, 505.0, 506.0, 507.0, 508.0, 509.0, 510.0, 511.0, 512.0, 513.0, 514.0, 515.0, 516.0, 517.0, 518.0, 519.0, 520.0, 521.0, 522.0, 523.0, 524.0, 525.0, 526.0, 527.0, 528.0, 529.0, 530.0, 531.0, 532.0, 533.0, 534.0, 535.0, 536.0, 537.0, 538.0, 539.0, 540.0, 541.0, 542.0, 543.0, 544.0, 545.0, 546.0, 547.0, 548.0, 549.0, 550.0, 551.0, 552.0, 553.0, 554.0, 555.0, 556.0, 557.0, 558.0, 559.0, 560.0, 561.0, 562.0, 563.0, 564.0, 565.0, 566.0, 567.0, 568.0, 569.0, 570.0, 571.0, 572.0, 573.0, 574.0, 575.0, 576.0, 577.0, 578.0, 579.0, 580.0, 581.0, 582.0, 583.0, 584.0, 585.0, 586.0, 587.0, 588.0, 589.0, 590.0, 591.0, 592.0, 593.0, 594.0, 595.0, 596.0, 597.0, 598.0, 599.0, 600.0, 601.0, 602.0, 603.0, 604.0, 605.0, 606.0, 607.0, 608.0, 609.0, 610.0, 611.0, 612.0, 613.0, 614.0, 615.0, 616.0, 617.0, 618.0, 619.0, 620.0, 621.0, 622.0, 623.0, 624.0, 625.0, 626.0, 627.0, 628.0, 629.0, 630.0, 631.0, 632.0, 633.0, 634.0, 635.0, 636.0, 637.0, 638.0, 639.0, 640.0, 641.0, 642.0, 643.0, 644.0, 645.0, 646.0, 647.0, 648.0, 649.0, 650.0, 651.0, 652.0, 653.0, 654.0, 655.0, 656.0, 657.0, 658.0, 659.0, 660.0, 661.0, 662.0, 663.0, 664.0, 665.0, 666.0, 667.0, 668.0, 669.0, 670.0, 671.0, 672.0, 673.0, 674.0, 675.0, 676.0, 677.0, 678.0, 679.0, 680.0, 681.0, 682.0, 683.0, 684.0, 685.0, 686.0, 687.0, 688.0, 689.0, 690.0, 691.0, 692.0, 693.0, 694.0, 695.0, 696.0, 697.0, 698.0, 699.0], "dtype": "float32"}, {"__nd__": [699.0, 698.0, 697.0, 696.0, 695.0, 694.0, 693.0, 692.0, 691.0, 690.0, 689.0, 688.0, 687.0, 686.0, 685.0, 684.0, 683.0, 682.0, 681.0, 680.0, 679.0, 678.0, 677.0, 676.0, 675.0, 674.0, 673.0, 672.0, 671.0, 670.0, 669.0, 668.0, 667.0, 666.0, 665.0, 664.0, 663.0, 662.0, 661.0, 660.0, 659.0, 658.0, 657.0, 656.0, 655.0, 654.0, 653.0, 652.0, 651.0, 650.0, 649.0, 648.0, 647.0, 646.0, 645.0, 644.0, 643.0, 642.0, 641.0, 640.0, 639.0, 638.0, 637.0, 636.0, 635.0, 634.0, 633.0, 632.0, 631.0, 630.0, 629.0, 628.0, 627.0, 626.0, 625.0, 624.0, 623.0, 622.0, 621.0, 620.0, 619.0, 618.0, 617.0, 616.0, 615.0, 614.0, 613.0, 612.0, 611.0, 610.0, 609.0, 608.0, 607.0, 606.0, 605.0, 604.0, 603.0, 602.0, 601.0, 600.0, 599.0, 598.0, 597.0, 596.0, 595.0, 594.0, 593.0, 592.0, 591.0, 590.0, 589.0, 588.0, 587.0, 586.0, 585.0, 584.0, 583.0, 582.0, 581.0, 580.0, 579.0, 578.0, 577.0, 576.0, 575.0, 574.0, 573.0, 572.0, 571.0, 570.0, 569.0, 568.0, 567.0, 566.0, 565.0, 564.0, 563.0, 562.0, 561.0, 560.0, 559.0, 558.0, 557.0, 556.0, 555.0, 554.0, 553.0, 552.0, 551.0, 550.0, 549.0, 548.0, 547.0, 546.0, 545.0, 544.0, 543.0, 542.0, 541.0, 540.0, 539.0, 538.0, 537.0, 536.0, 535.0, 534.0, 533.0, 532.0, 531.0, 530.0, 529.0, 528.0, 527.0, 526.0, 525.0, 524.0, 523.0, 522.0, 521.0, 520.0, 519.0, 518.0, 517.0, 516.0, 515.0, 514.0, 513.0, 512.0, 511.0, 510.0, 509.0, 508.0, 507.0, 506.0, 505.0, 504.0, 503.0, 502.0, 501.0, 500.0, 499.0, 498.0, 497.0, 496.0, 495.0, 494.0, 493.0, 492.0, 491.0, 490.0, 489.0, 488.0, 487.0, 486.0, 485.0, 484.0, 483.0, 482.0, 481.0, 480.0, 479.0, 478.0, 477.0, 476.0, 475.0, 474.0, 473.0, 472.0, 471.0, 470.0, 469.0, 468.0, 467.0, 466.0, 465.0, 464.0, 463.0, 462.0, 461.0, 460.0, 459.0, 458.0, 457.0, 456.0, 455.0, 454.0, 453.0, 452.0, 451.0, 450.0, 449.0, 448.0, 447.0, 446.0, 445.0, 444.0, 443.0, 442.0, 441.0, 440.0, 439.0, 438.0, 437.0, 436.0, 435.0, 434.0, 433.0, 432.0, 431.0, 430.0, 429.0, 428.0, 427.0, 426.0, 425.0, 424.0, 423.0, 422.0, 421.0, 420.0, 419.0, 418.0, 417.0, 416.0, 415.0, 414.0, 413.0, 412.0, 411.0, 410.0, 409.0, 408.0, 407.0, 406.0, 405.0, 404.0, 403.0, 402.0, 401.0, 400.0, 399.0, 398.0, 397.0, 396.0, 395.0, 394.0, 393.0, 392.0, 391.0, 390.0, 389.0, 388.0, 387.0, 386.0, 385.0, 384.0, 383.0, 382.0, 381.0, 380.0, 379.0, 378.0, 377.0, 376.0, 375.0, 374.0, 373.0, 372.0, 371.0, 370.0, 369.0, 368.0, 367.0, 366.0, 365.0, 364.0, 363.0, 362.0, 361.0, 360.0, 359.0, 358.0, 357.0, 356.0, 355.0, 354.0, 353.0, 352.0, 351.0, 350.0, 349.0, 348.0, 347.0, 346.0, 345.0, 344.0, 343.0, 342.0, 341.0, 340.0, 339.0, 338.0, 337.0, 336.0, 335.0, 334.0, 333.0, 332.0, 331.0, 330.0, 329.0, 328.0, 327.0, 326.0, 325.0, 324.0, 323.0, 322.0, 321.0, 320.0, 319.0, 318.0, 317.0, 316.0, 315.0, 314.0, 313.0, 312.0, 311.0, 310.0, 309.0, 308.0, 307.0, 306.0, 305.0, 304.0, 303.0, 302.0, 301.0, 300.0, 299.0, 298.0, 297.0, 296.0, 295.0, 294.0, 293.0, 292.0, 291.0, 290.0, 289.0, 288.0, 287.0, 286.0, 285.0, 284.0, 283.0, 282.0, 281.0, 280.0, 279.0, 278.0, 277.0, 276.0, 275.0, 274.0, 273.0, 272.0, 271.0, 270.0, 269.0, 268.0, 267.0, 266.0, 265.0, 264.0, 263.0, 262.0, 261.0, 260.0, 259.0, 258.0, 257.0, 256.0, 255.0, 254.0, 253.0, 252.0, 251.0, 250.0, 249.0, 248.0, 247.0, 246.0, 245.0, 244.0, 243.0, 242.0, 241.0, 240.0, 239.0, 238.0, 237.0, 236.0, 235.0, 234.0, 233.0, 232.0, 231.0, 230.0, 229.0, 228.0, 227.0, 226.0, 225.0, 224.0, 223.0, 222.0, 221.0, 220.0, 219.0, 218.0, 217.0, 216.0, 215.0, 214.0, 213.0, 212.0, 211.0, 210.0, 209.0, 208.0, 207.0, 206.0, 205.0, 204.0, 203.0, 202.0, 201.0, 200.0, 199.0, 198.0, 197.0, 196.0, 195.0, 194.0, 193.0, 192.0, 191.0, 190.0, 189.0, 188.0, 187.0, 186.0, 185.0, 184.0, 183.0, 182.0, 181.0, 180.0, 179.0, 178.0, 177.0, 176.0, 175.0, 174.0, 173.0, 172.0, 171.0, 170.0, 169.0, 168.0, 167.0, 166.0, 165.0, 164.0, 163.0, 162.0, 161.0, 160.0, 159.0, 158.0, 157.0, 156.0, 155.0, 154.0, 153.0, 152.0, 151.0, 150.0, 149.0, 148.0, 147.0, 146.0, 145.0, 144.0, 143.0, 142.0, 141.0, 140.0, 139.0, 138.0, 137.0, 136.0, 135.0, 134.0, 133.0, 132.0, 131.0, 130.0, 129.0, 128.0, 127.0, 126.0, 125.0, 124.0, 123.0, 122.0, 121.0, 120.0, 119.0, 118.0, 117.0, 116.0, 115.0, 114.0, 113.0, 112.0, 111.0, 110.0, 109.0, 108.0, 107.0, 106.0, 105.0, 104.0, 103.0, 102.0, 101.0, 100.0, 99.0, 98.0, 97.0, 96.0, 95.0, 94.0, 93.0, 92.0, 91.0, 90.0, 89.0, 88.0, 87.0, 86.0, 85.0, 84.0, 83.0, 82.0, 81.0, 80.0, 79.0, 78.0, 77.0, 76.0, 75.0, 74.0, 73.0, 72.0, 71.0, 70.0, 69.0, 68.0, 67.0, 66.0, 65.0, 64.0, 63.0, 62.0, 61.0, 60.0, 59.0, 58.0, 57.0, 56.0, 55.0, 54.0, 53.0, 52.0, 51.0, 50.0, 49.0, 48.0, 47.0, 46.0, 45.0, 44.0, 43.0, 42.0, 41.0, 40.0, 39.0, 38.0, 37.0, 36.0, 35.0, 34.0, 33.0, 32.0, 31.0, 30.0, 29.0, 28.0, 27.0, 26.0, 25.0, 24.0, 23.0, 22.0, 21.0, 20.0, 19.0, 18.0, 17.0, 16.0, 15.0, 14.0, 13.0, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0, 1.0, 0.0], "dtype": "float32"}], "e": {"__nd__": [-699.0, -697.0, -695.0, -693.0, -691.0, -689.0, -687.0, -685.0, -683.0, -681.0, -679.0, -677.0, -675.0, -673.0, -671.0, -669.0, -667.0, -665.0, -663.0, -661.0, -659.0, -657.0, -655.0, -653.0, -651.0, -649.0, -647.0, -645.0, -643.0, -641.0, -639.0, -637.0, -635.0, -633.0, -631.0, -629.0, -627.0, -625.0, -623.0, -621.0, -619.0, -617.0, -615.0, -613.0, -611.0, -609.0, -607.0, -605.0, -603.0, -601.0, -599.0, -597.0, -595.0, -593.0, -591.0, -589.0, -587.0, -585.0, -583.0, -581.0, -579.0, -577.0, -575.0, -573.0, -571.0, -569.0, -567.0, -565.0, -563.0, -561.0, -559.0, -557.0, -555.0, -553.0, -551.0, -549.0, -547.0, -545.0, -543.0, -541.0, -539.0, -537.0, -535.0, -533.0, -531.0, -529.0, -527.0, -525.0, -523.0, -521.0, -519.0, -517.0, -515.0, -513.0, -511.0, -509.0, -507.0, -505.0, -503.0, -501.0, -499.0, -497.0, -495.0, -493.0, -491.0, -489.0, -487.0, -485.0, -483.0, -481.0, -479.0, -477.0, -475.0, -473.0, -471.0, -469.0, -467.0, -465.0, -463.0, -461.0, -459.0, -457.0, -455.0, -453.0, -451.0, -449.0, -447.0, -445.0, -443.0, -441.0, -439.0, -437.0, -435.0, -433.0, -431.0, -429.0, -427.0, -425.0, -423.0, -421.0, -419.0, -417.0, -415.0, -413.0, -411.0, -409.0, -407.0, -405.0, -403.0, -401.0, -399.0, -397.0, -395.0, -393.0, -391.0, -389.0, -387.0, -385.0, -383.0, -381.0, -379.0, -377.0, -375.0, -373.0, -371.0, -369.0, -367.0, -365.0, -363.0, -361.0, -359.0, -357.0, -355.0, -353.0, -351.0, -349.0, -347.0, -345.0, -343.0, -341.0, -339.0, -337.0, -335.0, -333.0, -331.0, -329.0, -327.0, -325.0, -323.0, -321.0, -319.0, -317.0, -315.0, -313.0, -311.0, -309.0, -307.0, -305.0, -303.0, -301.0, -299.0, -297.0, -295.0, -293.0, -291.0, -289.0, -287.0, -285.0, -283.0, -281.0, -279.0, -277.0, -275.0, -273.0, -271.0, -269.0, -267.0, -265.0, -263.0, -261.0, -259.0, -257.0, -255.0, -253.0, -251.0, -249.0, -247.0, -245.0, -243.0, -241.0, -239.0, -237.0, -235.0, -233.0, -231.0, -229.0, -227.0, -225.0, -223.0, -221.0, -219.0, -217.0, -215.0, -213.0, -211.0, -209.0, -207.0, -205.0, -203.0, -201.0, -199.0, -197.0, -195.0, -193.0, -191.0, -189.0, -187.0, -185.0, -183.0, -181.0, -179.0, -177.0, -175.0, -173.0, -171.0, -169.0, -167.0, -165.0, -163.0, -161.0, -159.0, -157.0, -155.0, -153.0, -151.0, -149.0, -147.0, -145.0, -143.0, -141.0, -139.0, -137.0, -135.0, -133.0, -131.0, -129.0, -127.0, -125.0, -123.0, -121.0, -119.0, -117.0, -115.0, -113.0, -111.0, -109.0, -107.0, -105.0, -103.0, -101.0, -99.0, -97.0, -95.0, -93.0, -91.0, -89.0, -87.0, -85.0, -83.0, -81.0, -79.0, -77.0, -75.0, -73.0, -71.0, -69.0, -67.0, -65.0, -63.0, -61.0, -59.0, -57.0, -55.0, -53.0, -51.0, -49.0, -47.0, -45.0, -43.0, -41.0, -39.0, -37.0, -35.0, -33.0, -31.0, -29.0, -27.0, -25.0, -23.0, -21.0, -19.0, -17.0, -15.0, -13.0, -11.0, -9.0, -7.0, -5.0, -3.0, -1.0, 1.0, 3.0, 5.0, 7.0, 9.0, 11.0, 13.0, 15.0, 17.0, 19.0, 21.0, 23.0, 25.0, 27.0, 29.0, 31.0, 33.0, 35.0, 37.0, 39.0, 41.0, 43.0, 45.0, 47.0, 49.0, 51.0, 53.0, 55.0, 57.0, 59.0, 61.0, 63.0, 65.0, 67.0, 69.0, 71.0, 73.0, 75.0, 77.0, 79.0, 81.0, 83.0, 85.0, 87.0, 89.0, 91.0, 93.0, 95.0, 97.0, 99.0, 101.0, 103.0, 105.0, 107.0, 109.0, 111.0, 113.0, 115.0, 117.0, 119.0, 121.0, 123.0, 125.0, 127.0, 129.0, 131.0, 133.0, 135.0, 137.0, 139.0, 141.0, 143.0, 145.0, 147.0, 149.0, 151.0, 153.0, 155.0, 157.0, 159.0, 161.0, 163.0, 165.0, 167.0, 169.0, 171.0, 173.0, 175.0, 177.0, 179.0, 181.0, 183.0, 185.0, 187.0, 189.0, 191.0, 193.0, 195.0, 197.0, 199.0, 201.0, 203.0, 205.0, 207.0, 209.0, 211.0, 213.0, 215.0, 217.0, 219.0, 221.0, 223.0, 225.0, 227.0, 229.0, 231.0, 233.0, 235.0, 237.0, 239.0, 241.0, 243.0, 245.0, 247.0, 249.0, 251.0, 253.0, 255.0, 257.0, 259.0, 261.0, 263.0, 265.0, 267.0, 269.0, 271.0, 273.0, 275.0, 277.0, 279.0, 281.0, 283.0, 285.0, 287.0, 289.0, 291.0, 293.0, 295.0, 297.0, 299.0, 301.0, 303.0, 305.0, 307.0, 309.0, 311.0, 313.0, 315.0, 317.0, 319.0, 321.0, 323.0, 325.0, 327.0, 329.0, 331.0, 333.0, 335.0, 337.0, 339.0, 341.0, 343.0, 345.0, 347.0, 349.0, 351.0, 353.0, 355.0, 357.0, 359.0, 361.0, 363.0, 365.0, 367.0, 369.0, 371.0, 373.0, 375.0, 377.0, 379.0, 381.0, 383.0, 385.0, 387.0, 389.0, 391.0, 393.0, 395.0, 397.0, 399.0, 401.0, 403.0, 405.0, 407.0, 409.0, 411.0, 413.0, 415.0, 417.0, 419.0, 421.0, 423.0, 425.0, 427.0, 429.0, 431.0, 433.0, 435.0, 437.0, 439.0, 441.0, 443.0, 445.0, 447.0, 449.0, 451.0, 453.0, 455.0, 457.0, 459.0, 461.0, 463.0, 465.0, 467.0, 469.0, 471.0, 473.0, 475.0, 477.0, 479.0, 481.0, 483.0, 485.0, 487.0, 489.0, 491.0, 493.0, 495.0, 497.0, 499.0, 501.0, 503.0, 505.0, 507.0, 509.0, 511.0, 513.0, 515.0, 517.0, 519.0, 521.0, 523.0, 525.0, 527.0, 529.0, 531.0, 533.0, 535.0, 537.0, 539.0, 541.0, 543.0, 545.0, 547.0, 549.0, 551.0, 553.0, 555.0, 557.0, 559.0, 561.0, 563.0, 565.0, 567.0, 569.0, 571.0, 573.0, 575.0, 577.0, 579.0, 581.0, 583.0, 585.0, 587.0, 589.0, 591.0, 593.0, 595.0, 597.0, 599.0, 601.0, 603.0, 605.0, 607.0, 609.0, 611.0, 613.0, 615.0, 617.0, 619.0, 621.0, 623.0, 625.0, 627.0, 629.0, 631.0, 633.0, 635.0, 637.0, 639.0, 641.0, 643.0, 645.0, 647.0, 649.0, 651.0, 653.0, 655.0, 657.0, 659.0, 661.0, 663.0, 665.0, 667.0, 669.0, 671.0, 673.0, 675.0, 677.0, 679.0, 681.0, 683.0, 685.0, 687.0, 689.0, 691.0, 693.0, 695.0, 697.0, 699.0], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(vector_sub, json.loads(_TESTS), 'cuda', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")